# Counterfactual Experience Replay via Structural Causal Models for Financial Reinforcement Learning


> All bugs patched, RTX 3090 optimised (AMP + cudnn.benchmark + torch.compile),
> robust checkpointing, full 7-method experiment loop.


## 1. Dependencies

In [1]:
# FIX: pin gcastle to avoid NotearsNonlinear API breakage between versions
!pip install "gcastle==1.0.3" networkx scipy scikit-learn statsmodels yfinance -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib
matplotlib.use('Agg')           # headless -- safe on server
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings, gc, os, time, json, copy
warnings.filterwarnings('ignore')

from castle.algorithms import PC
try:
    from castle.algorithms import NotearsNonlinear
    _NOTEARS_OK = True
except ImportError:
    _NOTEARS_OK = False
    print("  [WARN] NotearsNonlinear not available -- will use PC fallback for neural SCM")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler   # mixed precision

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# FIX: RTX 3090 optimisations
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False   # faster, still reproducible per seed

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'AMP     : enabled (float16 forward, float32 master weights)')
print('OK  All imports successful')



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


2026-07-30 09:17:35,810 - /home/nmit/anaconda3/lib/python3.12/site-packages/castle/backend/__init__.py[line:36] - INFO: You can use `os.environ['CASTLE_BACKEND'] = backend` to set the backend(`pytorch` or `mindspore`).
2026-07-30 09:17:35,867 - /home/nmit/anaconda3/lib/python3.12/site-packages/castle/algorithms/__init__.py[line:36] - INFO: You are using ``pytorch`` as the backend.


PyTorch : 2.7.1+cu126
Device  : cuda
GPU     : NVIDIA GeForce RTX 3090
VRAM    : 25.3 GB
AMP     : enabled (float16 forward, float32 master weights)
OK  All imports successful


## 2. Configuration -- RTX 3090 (24 GB VRAM)

In [2]:
ASSET_UNIVERSE = {
    'US_Tech'    : ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META', 'AMZN'],
    'US_Finance' : ['JPM', 'GS', 'BAC', 'MS', 'BLK', 'C'],
    'Asian_Mkts' : ['TSM', '9984.T', '005930.KS', '7203.T', '6758.T', '000660.KS'],
    'EU_Mkts'    : ['ASML', 'SAP', 'SIE.DE', 'BAS.DE', 'AIR.PA', 'DTE.DE'],
    'Commodities': ['GLD', 'SLV', 'USO', 'DBA', 'PDBC', 'IAU'],
}
STOCK_TICKERS = [t for s in ASSET_UNIVERSE.values() for t in s]  # 30 tickers

TRAIN_START = '2018-01-01';  TRAIN_END = '2021-12-31'
VAL_START   = '2022-01-01';  VAL_END   = '2022-12-31'
TEST_START  = '2023-01-01';  TEST_END  = '2024-12-31'

N_TRIALS    = 10
N_EPISODES  = 100
INITIAL_BAL = 100_000
TRANS_COST  = 0.001

CF_RATIO      = 0.5
CQL_ALPHA     = 0.01
N_CF_ACTIONS  = 2
CF_BETA       = 0.9      # epistemic discount on counterfactual rewards

# RTX 3090: 24 GB VRAM -- generous buffer & batch
BUFFER_CAP  = 200_000
BATCH_SIZE  = 256

TARGET_UPDATE_FREQ = 10   # episodes between hard target updates (DQN)

CKPT_DIR = 'journal_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

print(f'Assets   : {len(STOCK_TICKERS)} tickers across {len(ASSET_UNIVERSE)} sectors')
print(f'Trials   : {N_TRIALS} x {N_EPISODES} episodes')
print(f'Buffer   : {BUFFER_CAP:,}  |  Batch: {BATCH_SIZE}')
print(f'Device   : {DEVICE}')
print(f'CF beta  : {CF_BETA}  |  CQL alpha: {CQL_ALPHA}')

# -- Extended experiments (Section 7.7 limitations follow-up) --------------
TC_SWEEP_BPS    = [5, 10, 20, 50]                    # 10bps == TRANS_COST (paper default)
TC_SWEEP_LEVELS = [bp / 10_000 for bp in TC_SWEEP_BPS]

ROLLING_SCM_REFIT_EVERY = 20     # episodes between linear-SCM re-estimation
ROLLING_SCM_WINDOW_FRAC = 0.5    # fraction of train_data used as the rolling window

PPO_LR         = 3e-4
PPO_GAMMA      = 0.99
PPO_GAE_LAMBDA = 0.95
PPO_CLIP_EPS   = 0.2
PPO_EPOCHS     = 4
PPO_MINIBATCH  = 64
PPO_ENT_COEF   = 0.01
PPO_VF_COEF    = 0.5

print(f'TC sweep : {TC_SWEEP_BPS} bps')
print(f'Rolling SCM: refit every {ROLLING_SCM_REFIT_EVERY} episodes, window={ROLLING_SCM_WINDOW_FRAC}')
print(f'PPO      : lr={PPO_LR}  clip={PPO_CLIP_EPS}  epochs/update={PPO_EPOCHS}')


Assets   : 30 tickers across 5 sectors
Trials   : 10 x 100 episodes
Buffer   : 200,000  |  Batch: 256
Device   : cuda
CF beta  : 0.9  |  CQL alpha: 0.01
TC sweep : [5, 10, 20, 50] bps
Rolling SCM: refit every 20 episodes, window=0.5
PPO      : lr=0.0003  clip=0.2  epochs/update=4


## 3. Data Loading & Feature Engineering (12 features)

In [3]:
class TradingDataLoader:
    FEATURE_COLS = [
        'returns', 'volatility', 'volume_ratio',
        'momentum_5', 'momentum_10', 'momentum_20',
        'rsi', 'bb_position', 'macd_diff',
        'atr_ratio', 'obv_signal', 'price_vs_sma50',
    ]

    def __init__(self, ticker, start, end):
        self.ticker = ticker
        self.start  = start
        self.end    = end

    def load(self):
        try:
            raw = yf.download(self.ticker, start=self.start, end=self.end,
                              progress=False, auto_adjust=True)
            if isinstance(raw.columns, pd.MultiIndex):
                raw.columns = raw.columns.droplevel(1)
            if len(raw) < 150:
                raise ValueError(f'Only {len(raw)} rows -- insufficient history')
        except Exception as e:
            print(f'  WARN  {self.ticker}: {e}')
            return None

        df    = raw.copy()
        close = df['Close'].squeeze()
        high  = df['High'].squeeze()
        low   = df['Low'].squeeze()
        vol   = df['Volume'].squeeze()

        df['returns']      = close.pct_change()
        df['volatility']   = df['returns'].rolling(20).std()
        df['volume_ma']    = vol.rolling(20).mean()
        df['volume_ratio'] = vol / (df['volume_ma'] + 1e-9)
        df['momentum_5']   = close.pct_change(5)
        df['momentum_10']  = close.pct_change(10)
        df['momentum_20']  = close.pct_change(20)

        delta = close.diff()
        gain  = delta.where(delta > 0, 0).rolling(14).mean()
        loss  = (-delta.where(delta < 0, 0)).rolling(14).mean()
        df['rsi'] = (100 - 100 / (1 + gain / (loss + 1e-9))) / 100.0

        bb_mid = close.rolling(20).mean()
        bb_std = close.rolling(20).std()
        df['bb_position'] = (close - (bb_mid - 2*bb_std)) / (4*bb_std + 1e-9)

        exp1 = close.ewm(span=12).mean()
        exp2 = close.ewm(span=26).mean()
        macd = exp1 - exp2
        df['macd_diff'] = (macd - macd.ewm(span=9).mean()) / (close + 1e-9)

        tr = pd.concat([high - low,
                        (high - close.shift()).abs(),
                        (low  - close.shift()).abs()], axis=1).max(axis=1)
        df['atr_ratio'] = tr.rolling(14).mean() / (close + 1e-9)

        obv = (np.sign(df['returns']) * vol).cumsum()
        df['obv_signal'] = obv.pct_change(10)

        sma50 = close.rolling(50).mean()
        df['price_vs_sma50'] = (close - sma50) / (sma50 + 1e-9)

        df = df[self.FEATURE_COLS + ['Close', 'Volume']].dropna()
        for col in self.FEATURE_COLS:
            lo = df[col].quantile(0.001)
            hi = df[col].quantile(0.999)
            df[col] = df[col].clip(lo, hi)

        return df

STATE_DIM  = len(TradingDataLoader.FEATURE_COLS)   # 12
STATE_COLS = TradingDataLoader.FEATURE_COLS
print(f'OK  TradingDataLoader ready  ({STATE_DIM} features)')


OK  TradingDataLoader ready  (12 features)


## 4. Causal Structure Learning -- PC and NOTEARS-MLP

In [4]:
CAUSAL_VARS = [
    'returns', 'volatility', 'volume_ratio',
    'momentum_5', 'rsi', 'bb_position', 'macd_diff',
    'atr_ratio', 'obv_signal', 'price_vs_sma50',
]

def learn_causal_structure(features_df, alpha=0.05, method='pc'):
    """
    Learn causal graph from financial time series.
    Returns (causal_matrix, var_names, scaled_data_df)
    """
    dm = features_df[CAUSAL_VARS].copy()
    dm['action']       = np.sign(features_df['momentum_5'])
    dm['returns_lag1'] = features_df['returns'].shift(1)
    dm = dm.dropna()

    var_names    = list(dm.columns)
    scaler       = StandardScaler()
    dm_scaled    = scaler.fit_transform(dm.values)
    dm_scaled_df = pd.DataFrame(dm_scaled, columns=var_names)

    if method == 'notears' and _NOTEARS_OK:
        try:
            model = NotearsNonlinear()
            model.learn(dm_scaled)
            graph   = model.causal_matrix
            n_edges = int(np.sum(graph != 0))
            print(f'    [NOTEARS-MLP] {len(var_names)} nodes, {n_edges} edges')
            return graph, var_names, dm_scaled_df
        except Exception as e:
            print(f'    [NOTEARS failed -> PC fallback]: {e}')

    # PC algorithm (default / fallback)
    pc    = PC(alpha=alpha)
    pc.learn(dm_scaled)
    graph = pc.causal_matrix
    print(f'    [PC alpha={alpha}] {len(var_names)} nodes, {int(np.sum(graph!=0))} edges')
    return graph, var_names, dm_scaled_df

print('OK  Causal structure learner ready (PC + NOTEARS-MLP with fallback)')


OK  Causal structure learner ready (PC + NOTEARS-MLP with fallback)


## 5. Structural Causal Model -- Linear and Neural SEMs

Pearl's 3-step do-calculus:
1. **Abduction** -- infer exogenous noise `U_i` from observations
2. **Action** -- intervene `do(action = a')`
3. **Prediction** -- propagate through causal graph in topological order


In [5]:
class StructuralCausalModel:
    def __init__(self, causal_graph, var_names, mode='linear'):
        self.graph       = causal_graph
        self.var_names   = var_names
        self.n           = len(var_names)
        self.mode        = mode
        self.models      = {}
        self._r2_scores  = {}
        self._topo_order = None

    def fit(self, data_matrix: pd.DataFrame):
        self._topo_order = self._topological_sort()
        for i, var in enumerate(self.var_names):
            parent_idx = [j for j in range(self.n) if self.graph[j, i] != 0]
            if not parent_idx:
                self.models[var] = {
                    'type': 'exogenous',
                    'mean': float(data_matrix.iloc[:, i].mean()),
                    'std' : float(data_matrix.iloc[:, i].std()) + 1e-8,
                }
                self._r2_scores[var] = None
            else:
                parent_names = [self.var_names[p] for p in parent_idx]
                X = data_matrix[parent_names].values
                y = data_matrix[var].values
                reg = (MLPRegressor(hidden_layer_sizes=(64, 32), activation='relu',
                                    max_iter=500, random_state=42,
                                    early_stopping=True, validation_fraction=0.1)
                       if self.mode == 'neural' else LinearRegression())
                reg.fit(X, y)
                y_pred    = reg.predict(X)
                residuals = y - y_pred
                self.models[var] = {
                    'type'         : 'endogenous',
                    'parent_names' : parent_names,
                    'reg'          : reg,
                    'residual_std' : float(np.std(residuals)) + 1e-8,
                }
                self._r2_scores[var] = float(r2_score(y, y_pred))

    def fit_report(self):
        endo = {k: v for k, v in self._r2_scores.items() if v is not None}
        if not endo:
            return {}
        return {'mean_r2': float(np.mean(list(endo.values()))),
                'min_r2' : float(min(endo.values())),
                'per_var': endo}

    def generate_counterfactual_state(self, observed_row: dict,
                                      intervention_var: str,
                                      intervention_value: float) -> dict:
        # Step 1 -- Abduction: recover exogenous noise
        noise = {}
        for var in self.var_names:
            m = self.models.get(var)
            if m is None:
                noise[var] = 0.
                continue
            if m['type'] == 'exogenous':
                noise[var] = observed_row.get(var, m['mean'])
            else:
                pv   = np.array([observed_row.get(p, 0.) for p in m['parent_names']]).reshape(1, -1)
                pred = float(m['reg'].predict(pv)[0])
                noise[var] = observed_row.get(var, pred) - pred

        # Step 2 -- Action: apply intervention
        cf = dict(observed_row)
        cf[intervention_var] = intervention_value   # FIX: was 'intervention_val' (typo)

        # Step 3 -- Prediction: propagate forward in topological order
        int_idx     = self.var_names.index(intervention_var)
        descendants = self._get_descendants(int_idx)
        for var in self._topo_order:
            if var == intervention_var:
                continue
            if self.var_names.index(var) not in descendants:
                continue
            m = self.models.get(var)
            if m and m['type'] == 'endogenous':
                pv   = np.array([cf.get(p, 0.) for p in m['parent_names']]).reshape(1, -1)
                pred = float(m['reg'].predict(pv)[0])
                cf[var] = pred + noise.get(var, 0.)
        return cf

    def _topological_sort(self):
        in_deg = {v: 0 for v in self.var_names}
        for i in range(self.n):
            for j in range(self.n):
                if self.graph[i, j] != 0:
                    in_deg[self.var_names[j]] += 1
        queue = [v for v, d in in_deg.items() if d == 0]
        order = []
        while queue:
            node = queue.pop(0)
            order.append(node)
            idx = self.var_names.index(node)
            for j in range(self.n):
                if self.graph[idx, j] != 0:
                    in_deg[self.var_names[j]] -= 1
                    if in_deg[self.var_names[j]] == 0:
                        queue.append(self.var_names[j])
        # FIX: fall back gracefully if graph has cycles (can happen with NOTEARS)
        return order if len(order) == self.n else list(self.var_names)

    def _get_descendants(self, node_idx):
        visited, queue = set(), [node_idx]
        while queue:
            cur = queue.pop(0)
            for j in range(self.n):
                if self.graph[cur, j] != 0 and j not in visited:
                    visited.add(j)
                    queue.append(j)
        return visited

print('OK  StructuralCausalModel (linear + neural, do-calculus) ready')


OK  StructuralCausalModel (linear + neural, do-calculus) ready


## 6. Trading Environment

In [6]:
class TradingEnvironment:
    def __init__(self, data, initial_balance=INITIAL_BAL, tc=TRANS_COST):

        # Convert once to fast NumPy array
        if isinstance(data, pd.DataFrame):
            self.data = data[STATE_COLS + ['Close']].values.astype(np.float32)
        else:
            self.data = data.astype(np.float32)

        self.init_bal = initial_balance
        self.tc       = tc
        self.reset()

    def reset(self):
        self.step_idx = 0
        self.balance  = self.init_bal
        self.shares   = 0.0
        self.pv       = self.init_bal
        self.history  = []
        return self._state()

    def _state(self):

        if self.step_idx >= len(self.data):
            return np.zeros(STATE_DIM, dtype=np.float32)

        # Fast NumPy indexing instead of pandas iloc
        s = self.data[self.step_idx][:STATE_DIM]

        return np.nan_to_num(
            s,
            nan=0.0,
            posinf=1.0,
            neginf=-1.0
        ).astype(np.float32)

    def step(self, action):

        if self.step_idx >= len(self.data) - 1:
            return self._state(), 0.0, True

        # Close price is last column
        price      = float(self.data[self.step_idx][-1])
        next_price = float(self.data[self.step_idx + 1][-1])

        # Buy
        if action == 2 and self.balance > 0:
            self.shares += self.balance / (price * (1 + self.tc))
            self.balance = 0.0

        # Sell
        elif action == 0 and self.shares > 0:
            self.balance += self.shares * price * (1 - self.tc)
            self.shares = 0.0

        old_pv  = self.pv
        self.pv = self.balance + self.shares * next_price

        reward = (self.pv - old_pv) / (old_pv + 1e-9)

        self.history.append({
            'step': self.step_idx,
            'action': action,
            'pv': self.pv,
            'reward': reward
        })

        self.step_idx += 1

        done = (self.step_idx >= len(self.data) - 1)

        return self._state(), reward, done

    def metrics(self):

        if not self.history:
            return {
                'total_return': 0.0,
                'sharpe_ratio': 0.0,
                'sortino_ratio': 0.0,
                'calmar_ratio': 0.0,
                'max_drawdown': 0.0,
                'win_rate': 0.0,
                'turnover': 0.0,
                'final_value': 0.0
            }

        h = pd.DataFrame(self.history)

        rets = h['reward'].values
        pv   = h['pv'].values

        tr = (self.pv - self.init_bal) / self.init_bal

        sh = (
            np.mean(rets) /
            (np.std(rets) + 1e-9)
        ) * np.sqrt(252)

        neg = rets[rets < 0]

        so = (
            np.mean(rets) /
            (np.std(neg) + 1e-9)
        ) * np.sqrt(252) if len(neg) > 0 else sh

        peak = np.maximum.accumulate(pv)

        mdd = float(
            np.min((pv - peak) / (peak + 1e-9))
        )

        ann = tr * 252 / max(len(rets), 1)

        cal = ann / (abs(mdd) + 1e-9)

        return {
            'total_return': tr,
            'sharpe_ratio': sh,
            'sortino_ratio': so,
            'calmar_ratio': cal,
            'max_drawdown': mdd,
            'win_rate': float(np.mean(rets > 0)),
            'turnover': float(np.mean(h['action'] != 1)),
            'final_value': self.pv
        }

print(f'OK  TradingEnvironment ready  (state_dim={STATE_DIM})')

OK  TradingEnvironment ready  (state_dim=12)


## 7. Networks & Replay Buffers

In [7]:
class DuelingQNet(nn.Module):
    def __init__(self, state_dim=STATE_DIM, action_dim=3, hidden=(512, 256)):
        super().__init__()
        layers, in_d = [], state_dim
        for h in hidden:
            layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.ReLU(), nn.Dropout(0.1)]
            in_d = h
        self.shared = nn.Sequential(*layers)
        self.value  = nn.Sequential(nn.Linear(in_d, 128), nn.ReLU(), nn.Linear(128, 1))
        self.adv    = nn.Sequential(nn.Linear(in_d, 128), nn.ReLU(), nn.Linear(128, action_dim))
        self.float()

    def forward(self, x):
        h = self.shared(x)
        v = self.value(h)
        a = self.adv(h)
        return v + (a - a.mean(dim=1, keepdim=True))


class SACActorDiscrete(nn.Module):
    def __init__(self, state_dim=STATE_DIM, action_dim=3, hidden=(512, 256)):
        super().__init__()
        layers, in_d = [], state_dim
        for h in hidden:
            layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.ReLU()]
            in_d = h
        layers.append(nn.Linear(in_d, action_dim))
        self.net = nn.Sequential(*layers)
        self.float()

    def forward(self, x):
        return F.softmax(self.net(x), dim=-1)

    def get_action(self, x):
        probs    = self.forward(x)
        dist     = torch.distributions.Categorical(probs)
        action   = dist.sample()
        log_prob = torch.log(probs + 1e-8)
        entropy  = -(probs * log_prob).sum(dim=-1, keepdim=True)
        return action, probs, log_prob, entropy


class SACCriticDiscrete(nn.Module):
    def __init__(self, state_dim=STATE_DIM, action_dim=3, hidden=(512, 256)):
        super().__init__()
        def mlp():
            layers, in_d = [], state_dim
            for h in hidden:
                layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.ReLU()]
                in_d = h
            layers.append(nn.Linear(in_d, action_dim))
            return nn.Sequential(*layers)
        self.q1 = mlp()
        self.q2 = mlp()
        self.float()

    def forward(self, x):
        return self.q1(x), self.q2(x)


# FIX: replace=True when buffer smaller than batch to avoid crash on early training
class PrioritizedReplayBuffer:
    def __init__(self, capacity=BUFFER_CAP, alpha=0.6,
                 beta_start=0.4, beta_end=1.0, beta_steps=100_000):
        self.capacity  = capacity
        self.alpha     = alpha
        self.beta      = beta_start
        self.beta_end  = beta_end
        self.beta_inc  = (beta_end - beta_start) / beta_steps
        self.buf       = []
        self.priorities = np.zeros(capacity, dtype=np.float32)
        self.ptr       = 0
        self.max_prio  = 1.0

    def push(self, *t):
        if len(self.buf) < self.capacity:
            self.buf.append(t)
        else:
            self.buf[self.ptr] = t
        self.priorities[self.ptr] = self.max_prio
        self.ptr = (self.ptr + 1) % self.capacity

    def sample(self, n):
        buf_len = len(self.buf)
        n       = min(n, buf_len)
        prios   = self.priorities[:buf_len] ** self.alpha
        probs   = prios / prios.sum()
        # FIX: use replace=True when buf smaller than requested n (early training)
        replace = (buf_len < n)
        idx     = np.random.choice(buf_len, n, replace=replace, p=probs)
        w       = (buf_len * probs[idx]) ** (-self.beta)
        w      /= w.max() + 1e-8
        self.beta = min(self.beta_end, self.beta + self.beta_inc)
        return [self.buf[i] for i in idx], idx, w

    def update_priorities(self, indices, td_errors):
        for i, e in zip(indices, td_errors):
            p = float(abs(e)) + 1e-6
            self.priorities[i] = p
            self.max_prio = max(self.max_prio, p)

    def __len__(self): return len(self.buf)


class UniformReplayBuffer:
    def __init__(self, capacity=BUFFER_CAP):
        self.capacity = capacity
        self.buf = []
        self.ptr = 0

    def push(self, *t):
        if len(self.buf) < self.capacity:
            self.buf.append(t)
        else:
            self.buf[self.ptr] = t
            self.ptr = (self.ptr + 1) % self.capacity

    def sample(self, n):
        n   = min(n, len(self.buf))
        idx = np.random.choice(len(self.buf), n, replace=False)
        return [self.buf[i] for i in idx]

    def __len__(self): return len(self.buf)

print('OK  DuelingQNet, SAC nets, PrioritizedReplayBuffer, UniformReplayBuffer ready')


OK  DuelingQNet, SAC nets, PrioritizedReplayBuffer, UniformReplayBuffer ready


In [8]:
class PPOActorCritic(nn.Module):
    """Shared-trunk actor-critic for discrete PPO."""
    def __init__(self, state_dim=STATE_DIM, action_dim=3, hidden=(512, 256)):
        super().__init__()
        layers, in_d = [], state_dim
        for h in hidden:
            layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.ReLU()]
            in_d = h
        self.shared      = nn.Sequential(*layers)
        self.policy_head = nn.Linear(in_d, action_dim)
        self.value_head  = nn.Linear(in_d, 1)
        self.float()

    def forward(self, x):
        h      = self.shared(x)
        logits = self.policy_head(h)
        value  = self.value_head(h).squeeze(-1)
        return logits, value

    def get_action_and_value(self, x, action=None):
        logits, value = self.forward(x)
        dist = torch.distributions.Categorical(logits=logits)
        if action is None:
            action = dist.sample()
        return action, dist.log_prob(action), dist.entropy(), value


class PPORolloutBuffer:
    """On-policy buffer -- cleared after every update. This is the architectural
    difference noted in Section 7.7 limitation #1: PPO has no persistent replay
    buffer for CER's counterfactual transitions to live in."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.states, self.actions, self.log_probs = [], [], []
        self.rewards, self.dones, self.values = [], [], []

    def push(self, s, a, logp, r, d, v):
        self.states.append(s);    self.actions.append(a)
        self.log_probs.append(logp); self.rewards.append(r)
        self.dones.append(d);     self.values.append(v)

    def __len__(self):
        return len(self.states)

print('OK  PPOActorCritic + PPORolloutBuffer ready')


OK  PPOActorCritic + PPORolloutBuffer ready


## 8. Agents -- Dueling DQN + SAC + CER Mixin

In [9]:
def batch_to_tensors(batch, device):
    s  = torch.tensor(np.array([b[0] for b in batch]), dtype=torch.float32).to(device)
    a  = torch.tensor(np.array([b[1] for b in batch]), dtype=torch.long).to(device)
    r  = torch.tensor(np.array([b[2] for b in batch]), dtype=torch.float32).to(device)
    s2 = torch.tensor(np.array([b[3] for b in batch]), dtype=torch.float32).to(device)
    d  = torch.tensor(np.array([b[4] for b in batch]), dtype=torch.float32).to(device)
    return s, a, r, s2, d


# ── Dueling Double DQN (baseline) ────────────────────────────────────────────
class DuelingDQNAgent:
    def __init__(self, lr=3e-4, gamma=0.99, device=DEVICE):
        self.gamma    = gamma
        self.device   = device
        self.q_net    = DuelingQNet().to(device).float()
        self.tgt_net  = DuelingQNet().to(device).float()
        self.tgt_net.load_state_dict(self.q_net.state_dict())
        self.optim    = optim.Adam(self.q_net.parameters(), lr=lr, weight_decay=1e-5)
        self.memory   = PrioritizedReplayBuffer(BUFFER_CAP)
        self.batch    = BATCH_SIZE
        self.scaler   = GradScaler()     # AMP gradient scaler

    def act(self, state, eps=0.1):
        if np.random.random() < eps:
            return np.random.randint(3)
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.device)
            
            return int(self.q_net(s).argmax())

    def push(self, *t): self.memory.push(*t)

    def train_step(self):
        if len(self.memory) < self.batch:
            return 0.
        batch, idx, w = self.memory.sample(self.batch)
        return self._update(batch, idx, w)

    def _update(self, batch, idx=None, w=None):
        s, a, r, s2, d = batch_to_tensors(batch, self.device)
        n  = len(batch)
        wt = torch.FloatTensor(w[:n]).to(self.device) if w is not None              else torch.ones(n, device=self.device)

        with autocast():
            q_cur = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze()
            with torch.no_grad():
                next_a = self.q_net(s2).argmax(1)
                q_next = self.tgt_net(s2).gather(1, next_a.unsqueeze(1)).squeeze()
                q_tgt  = r + self.gamma * q_next * (1 - d)
            td_err = (q_cur - q_tgt).detach()
            loss   = (wt * F.smooth_l1_loss(q_cur, q_tgt, reduction='none')).mean()

        self.optim.zero_grad()
        self.scaler.scale(loss).backward()
        self.scaler.unscale_(self.optim)
        nn.utils.clip_grad_norm_(self.q_net.parameters(), 10.)
        self.scaler.step(self.optim)
        self.scaler.update()

        if idx is not None:
            self.memory.update_priorities(idx, td_err.cpu().numpy())
        return float(loss)

    # FIX: hard update only -- called from training loop every TARGET_UPDATE_FREQ episodes
    def sync_target(self):
        self.tgt_net.load_state_dict(self.q_net.state_dict())


# ── Discrete SAC ─────────────────────────────────────────────────────────────
class DiscreteSACAgent:
    def __init__(self, lr=3e-4, gamma=0.99, alpha=0.2,
                 auto_alpha=True, target_entropy_ratio=0.98, device=DEVICE):
        self.gamma  = gamma
        self.device = device
        self.actor = SACActorDiscrete().to(device).float()
        self.critic = SACCriticDiscrete().to(device).float()
        self.critic_target = SACCriticDiscrete().to(device).float()
        self.critic_target.load_state_dict(self.critic.state_dict())
        self.actor_opt  = optim.Adam(self.actor.parameters(),  lr=lr)
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=lr)
        self.memory     = UniformReplayBuffer(BUFFER_CAP)
        self.batch      = BATCH_SIZE
        self.scaler     = GradScaler()

        self.auto_alpha = auto_alpha
        if auto_alpha:
            self.target_entropy = -np.log(1.0 / 3) * target_entropy_ratio
            self.log_alpha = torch.zeros(1, requires_grad=True, device=device)
            self.alpha     = self.log_alpha.exp().item()
            self.alpha_opt = optim.Adam([self.log_alpha], lr=lr)
        else:
            self.alpha = alpha

    def act(self, state, eps=0.):
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.device)
            
            return int(self.actor(s).argmax())

    def push(self, *t): self.memory.push(*t)

    def train_step(self):
        if len(self.memory) < self.batch:
            return 0.
        batch = self.memory.sample(self.batch)
        s, a, r, s2, d = batch_to_tensors(batch, self.device)

        with autocast():
            with torch.no_grad():
                _, probs2, log_probs2, _ = self.actor.get_action(s2)
                q1_next, q2_next = self.critic_target(s2)
                q_next   = torch.min(q1_next, q2_next)
                v_next   = (probs2 * (q_next - self.alpha * log_probs2)).sum(1)
                q_target = r + self.gamma * v_next * (1 - d)

            q1, q2 = self.critic(s)
            q1_a   = q1.gather(1, a.unsqueeze(1)).squeeze()
            q2_a   = q2.gather(1, a.unsqueeze(1)).squeeze()
            c_loss = F.mse_loss(q1_a, q_target) + F.mse_loss(q2_a, q_target)

        self.critic_opt.zero_grad()
        self.scaler.scale(c_loss).backward()
        self.scaler.unscale_(self.critic_opt)
        nn.utils.clip_grad_norm_(self.critic.parameters(), 10.)
        self.scaler.step(self.critic_opt)
        self.scaler.update()

        with autocast():
            _, probs, log_probs, _ = self.actor.get_action(s)
            q1_pi, q2_pi = self.critic(s)
            q_pi   = torch.min(q1_pi, q2_pi)
            a_loss = (probs * (self.alpha * log_probs - q_pi)).sum(1).mean()

        self.actor_opt.zero_grad()
        self.scaler.scale(a_loss).backward()
        self.scaler.unscale_(self.actor_opt)
        nn.utils.clip_grad_norm_(self.actor.parameters(), 10.)
        self.scaler.step(self.actor_opt)
        self.scaler.update()

        if self.auto_alpha:
            with autocast():
                alpha_loss = -(
                        self.log_alpha *
                        (log_probs.detach() + self.target_entropy) *
                        probs.detach()
                    ).sum(1).mean()
            self.alpha_opt.zero_grad()
            self.scaler.scale(alpha_loss).backward()
            self.scaler.step(self.alpha_opt)
            self.scaler.update()
            self.alpha = self.log_alpha.exp().item()

        for p, tp in zip(self.critic.parameters(), self.critic_target.parameters()):
            tp.data.copy_(0.005 * p.data + 0.995 * tp.data)

        return float(c_loss)

    def sync_target(self): pass  # SAC uses soft updates inside train_step


# ── CER Mixin ────────────────────────────────────────────────────────────────
class CERMixin:
    def init_cer(self, scm, var_names, cql_alpha=CQL_ALPHA):
        self.scm           = scm
        self.var_names     = var_names
        self.cql_alpha     = cql_alpha
        self.cf_memory     = UniformReplayBuffer(BUFFER_CAP)
        self.state_var_map = {v: i for i, v in enumerate(STATE_COLS) if v in var_names}

    def _state_to_obs(self, state):
        obs = {v: float(state[i]) for v, i in self.state_var_map.items()}
        for v in self.var_names:
            if v not in obs:
                obs[v] = 0.
        return obs

    def _obs_to_state(self, obs, original_state):
        cf = original_state.copy()
        for v, i in self.state_var_map.items():
            if v in obs:
                cf[i] = float(np.nan_to_num(obs[v], nan=0., posinf=1., neginf=-1.))
        return cf

    @staticmethod
    def _a2scm(a): return {0: -1., 1: 0., 2: 1.}[a]

    def generate_and_store_counterfactuals(self, state, action, reward, next_state, done):
        obs = self._state_to_obs(state)
        if 'action' in self.var_names:
            obs['action'] = self._a2scm(action)
        for alt_a in [a for a in range(3) if a != action][:N_CF_ACTIONS]:
            cf_obs   = self.scm.generate_counterfactual_state(obs, 'action', self._a2scm(alt_a))
            cf_state = self._obs_to_state(cf_obs, state)
            cf_r_raw = cf_obs.get('returns', reward)
            cf_r     = float(np.nan_to_num(cf_r_raw, nan=0., posinf=0.1, neginf=-0.1)) * CF_BETA
            self.cf_memory.push(cf_state, alt_a, cf_r, next_state, done)


# ── CausalDQN: Dueling DQN + PER + CER + Variance-Penalized CQL ──────────────
class CausalDQNAgent(DuelingDQNAgent, CERMixin):
    def __init__(self, scm, var_names, **kwargs):
        super().__init__(**kwargs)
        self.init_cer(scm, var_names)

    def train_step(self):
        if len(self.memory) < self.batch:
            return 0.
        real_n          = self.batch // 2
        cf_n            = self.batch - real_n
        real_b, idx, w  = self.memory.sample(real_n)

        if len(self.cf_memory) >= cf_n:
            cf_b  = self.cf_memory.sample(cf_n)
            mixed = real_b + cf_b
            # FIX: w only covers real_n samples; extend with ones for CF
            w_full = np.concatenate([w, np.ones(cf_n)])
        else:
            mixed  = real_b
            w_full = w

        s, a, r, s2, d = batch_to_tensors(mixed, self.device)
        n  = len(mixed)
        wt = torch.FloatTensor(w_full[:n]).to(self.device)

        with autocast():
            q_cur = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze()
            with torch.no_grad():
                next_a = self.q_net(s2).argmax(1)
                q_all  = self.tgt_net(s2)
                q_next = q_all.gather(1, next_a.unsqueeze(1)).squeeze()
                q_std  = q_all.std(dim=1)
                # Variance-Penalized CQL target
                q_cons = q_next - self.cql_alpha * q_std
                q_tgt  = r + self.gamma * q_cons * (1 - d)
            td_err = (q_cur - q_tgt).detach()
            loss   = (wt * F.smooth_l1_loss(q_cur, q_tgt, reduction='none')).mean()

        self.optim.zero_grad()
        self.scaler.scale(loss).backward()
        self.scaler.unscale_(self.optim)
        nn.utils.clip_grad_norm_(self.q_net.parameters(), 10.)
        self.scaler.step(self.optim)
        self.scaler.update()

        # FIX: only update PER priorities for the real_n real transitions
        self.memory.update_priorities(idx, td_err[:real_n].cpu().numpy())
        return float(loss)


# ── CausalSAC: SAC + CER ──────────────────────────────────────────────────────
class CausalSACAgent(DiscreteSACAgent, CERMixin):
    def __init__(self, scm, var_names, **kwargs):
        super().__init__(**kwargs)
        self.init_cer(scm, var_names)

    def train_step(self):
        # Mix real + CF transitions, then call SAC update on the mixed batch
        real_n = self.batch // 2
        cf_n   = self.batch - real_n
        if len(self.memory) < real_n:
            return 0.

        real_b = self.memory.sample(real_n)
        if len(self.cf_memory) >= cf_n:
            mixed = real_b + self.cf_memory.sample(cf_n)
        else:
            mixed = real_b

        s, a, r, s2, d = batch_to_tensors(mixed, self.device)

        with autocast():
            with torch.no_grad():
                _, probs2, log_probs2, _ = self.actor.get_action(s2)
                q1_next, q2_next = self.critic_target(s2)
                q_next   = torch.min(q1_next, q2_next)
                v_next   = (probs2 * (q_next - self.alpha * log_probs2)).sum(1)
                q_target = r + self.gamma * v_next * (1 - d)

            q1, q2 = self.critic(s)
            q1_a   = q1.gather(1, a.unsqueeze(1)).squeeze()
            q2_a   = q2.gather(1, a.unsqueeze(1)).squeeze()
            c_loss = F.mse_loss(q1_a, q_target) + F.mse_loss(q2_a, q_target)

        self.critic_opt.zero_grad()
        self.scaler.scale(c_loss).backward()
        self.scaler.unscale_(self.critic_opt)
        nn.utils.clip_grad_norm_(self.critic.parameters(), 10.)
        self.scaler.step(self.critic_opt)
        self.scaler.update()

        with autocast():
            _, probs, log_probs, _ = self.actor.get_action(s)
            q1_pi, q2_pi = self.critic(s)
            q_pi   = torch.min(q1_pi, q2_pi)
            a_loss = (probs * (self.alpha * log_probs - q_pi)).sum(1).mean()

        self.actor_opt.zero_grad()
        self.scaler.scale(a_loss).backward()
        self.scaler.unscale_(self.actor_opt)
        nn.utils.clip_grad_norm_(self.actor.parameters(), 10.)
        self.scaler.step(self.actor_opt)
        self.scaler.update()

        if self.auto_alpha:
            with autocast():
                alpha_loss = -(
                        self.log_alpha *
                        (log_probs.detach() + self.target_entropy) *
                        probs.detach()
                    ).sum(1).mean()
            self.alpha_opt.zero_grad()
            self.scaler.scale(alpha_loss).backward()
            self.scaler.step(self.alpha_opt)
            self.scaler.update()
            self.alpha = self.log_alpha.exp().item()

        for p, tp in zip(self.critic.parameters(), self.critic_target.parameters()):
            tp.data.copy_(0.005 * p.data + 0.995 * tp.data)

        return float(c_loss)

print('OK  DuelingDQN, SAC, CausalDQN (AMP), CausalSAC (AMP) agents ready')


OK  DuelingDQN, SAC, CausalDQN (AMP), CausalSAC (AMP) agents ready


In [10]:
class PPOAgent:
    """Discrete PPO -- an on-policy backbone baseline (Section 7.7 limitation #1).
    No persistent replay buffer and no CER counterfactual augmentation: on-policy
    methods discard trajectories after each update, so CER's replay-based design
    does not transfer without separate architectural treatment. This is a plain
    additional baseline family, evaluated with the same matched infrastructure
    (state/action space, environment, evaluation metrics) as every other method."""

    def __init__(self, lr=PPO_LR, gamma=PPO_GAMMA, gae_lambda=PPO_GAE_LAMBDA,
                 clip_eps=PPO_CLIP_EPS, epochs=PPO_EPOCHS, minibatch=PPO_MINIBATCH,
                 ent_coef=PPO_ENT_COEF, vf_coef=PPO_VF_COEF, device=DEVICE):
        self.gamma, self.gae_lambda = gamma, gae_lambda
        self.clip_eps, self.epochs, self.minibatch = clip_eps, epochs, minibatch
        self.ent_coef, self.vf_coef = ent_coef, vf_coef
        self.device = device
        self.net    = PPOActorCritic().to(device).float()
        self.optim  = optim.Adam(self.net.parameters(), lr=lr)
        self.buf    = PPORolloutBuffer()
        self.scaler = GradScaler()

    def act(self, state, eps=0.):
        """Greedy action -- kept eps-compatible so evaluate_agent/evaluate_on_val
        (shared with every other agent type) work unchanged."""
        s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits, _ = self.net(s)
            if eps > 0 and np.random.random() < eps:
                return int(np.random.randint(3))
            action = torch.argmax(logits, dim=-1)
        return int(action.item())

    def act_train(self, state):
        """Stochastic action + log-prob + value, for on-policy rollout collection."""
        s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.device)
        with torch.no_grad():
            action, log_prob, _, value = self.net.get_action_and_value(s)
        return int(action.item()), float(log_prob.item()), float(value.item())

    def store(self, s, a, logp, r, d, v):
        self.buf.push(s, a, logp, r, d, v)

    def _compute_gae(self, last_value):
        rewards = self.buf.rewards
        values  = self.buf.values + [last_value]
        dones   = self.buf.dones
        adv, gae = [0.] * len(rewards), 0.
        for t in reversed(range(len(rewards))):
            delta  = rewards[t] + self.gamma * values[t + 1] * (1 - dones[t]) - values[t]
            gae    = delta + self.gamma * self.gae_lambda * (1 - dones[t]) * gae
            adv[t] = gae
        adv = np.array(adv, dtype=np.float32)
        ret = adv + np.array(values[:-1], dtype=np.float32)
        return adv, ret

    def update(self, last_value=0.):
        if len(self.buf) == 0:
            return 0.
        adv, ret = self._compute_gae(last_value)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        s        = torch.tensor(np.array(self.buf.states),  dtype=torch.float32).to(self.device)
        a        = torch.tensor(np.array(self.buf.actions), dtype=torch.long).to(self.device)
        old_logp = torch.tensor(np.array(self.buf.log_probs), dtype=torch.float32).to(self.device)
        adv_t    = torch.tensor(adv, dtype=torch.float32).to(self.device)
        ret_t    = torch.tensor(ret, dtype=torch.float32).to(self.device)

        n       = len(self.buf)
        idx_all = np.arange(n)
        total_loss = 0.
        for _ in range(self.epochs):
            np.random.shuffle(idx_all)
            for start in range(0, n, self.minibatch):
                mb     = idx_all[start:start + self.minibatch]
                mb_idx = torch.tensor(mb, dtype=torch.long).to(self.device)
                with autocast():
                    _, new_logp, entropy, value = self.net.get_action_and_value(s[mb_idx], a[mb_idx])
                    ratio  = torch.exp(new_logp - old_logp[mb_idx])
                    surr1  = ratio * adv_t[mb_idx]
                    surr2  = torch.clamp(ratio, 1 - self.clip_eps, 1 + self.clip_eps) * adv_t[mb_idx]
                    pg_loss  = -torch.min(surr1, surr2).mean()
                    v_loss   = F.mse_loss(value, ret_t[mb_idx])
                    ent_loss = -entropy.mean()
                    loss = pg_loss + self.vf_coef * v_loss + self.ent_coef * ent_loss

                self.optim.zero_grad()
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optim)
                nn.utils.clip_grad_norm_(self.net.parameters(), 0.5)
                self.scaler.step(self.optim)
                self.scaler.update()
                total_loss += float(loss)

        self.buf.reset()
        return total_loss

    def sync_target(self):
        pass  # no target network in PPO

print('OK  PPOAgent (discrete, GAE, clipped surrogate) ready')


OK  PPOAgent (discrete, GAE, clipped surrogate) ready


## 9. Ablation Variants

In [11]:
# FIX: RandomCFDQNAgent -- correct MRO; call DuelingDQNAgent.__init__ explicitly
class RandomCFDQNAgent(DuelingDQNAgent):
    """Ablation: random Gaussian perturbation instead of SCM counterfactuals."""
    def __init__(self, noise_scale=0.05, **kwargs):
        DuelingDQNAgent.__init__(self, **kwargs)
        self.cf_memory   = UniformReplayBuffer(BUFFER_CAP)
        self.noise_scale = noise_scale
        self.cql_alpha   = CQL_ALPHA   # keep CQL for fair comparison

    def generate_and_store_counterfactuals(self, state, action, reward, next_state, done):
        for alt_a in [a for a in range(3) if a != action][:N_CF_ACTIONS]:
            cf_s = (state + np.random.normal(0, self.noise_scale,
                                             state.shape)).astype(np.float32)
            cf_r = float(reward + np.random.normal(0, self.noise_scale))
            self.cf_memory.push(cf_s, alt_a, cf_r, next_state, done)

    def train_step(self):
        if len(self.memory) < self.batch:
            return 0.
        real_n          = self.batch // 2
        real_b, idx, w  = self.memory.sample(real_n)
        if len(self.cf_memory) >= self.batch - real_n:
            mixed  = real_b + self.cf_memory.sample(self.batch - real_n)
            w_full = np.concatenate([w, np.ones(self.batch - real_n)])
        else:
            mixed  = real_b
            w_full = w
        return self._update(mixed, idx, w_full)


class NoCQLDQNAgent(CausalDQNAgent):
    """Ablation: CER without CQL conservative penalty (cql_alpha = 0)."""
    def __init__(self, scm, var_names, **kwargs):
        super().__init__(scm, var_names, **kwargs)
        self.cql_alpha = 0.0


class NoPERDQNAgent(CausalDQNAgent):
    """Ablation: CER + CQL, uniform replay (no PER)."""
    def __init__(self, scm, var_names, **kwargs):
        super().__init__(scm, var_names, **kwargs)
        self.memory = UniformReplayBuffer(BUFFER_CAP)   # replace PER with uniform

    def train_step(self):
        if len(self.memory) < self.batch:
            return 0.
        real_n = self.batch // 2
        real_b = self.memory.sample(real_n)
        if len(self.cf_memory) >= self.batch - real_n:
            mixed = real_b + self.cf_memory.sample(self.batch - real_n)
        else:
            mixed = real_b
        # No IS weights for uniform buffer -- pass w=None
        return self._update(mixed, idx=None, w=None)

print('OK  Ablation agents ready: RandomCF, NoCQL, NoPER')


OK  Ablation agents ready: RandomCF, NoCQL, NoPER


In [12]:
def refit_linear_scm_rolling(train_data, window_frac=ROLLING_SCM_WINDOW_FRAC):
    """Re-estimate the linear SCM on the most recent window_frac of train_data,
    simulating periodic re-estimation as new data arrives (Section 7.7 limitation
    #3: a fixed SCM may go stale mid-deployment). Uses PC (not NOTEARS-MLP) since
    it needs to run many times cheaply during a single training run."""
    n      = len(train_data)
    w      = max(int(n * window_frac), 60)
    window = train_data.iloc[-w:]
    graph, vn, dm = learn_causal_structure(window, method='pc')
    scm = StructuralCausalModel(graph, vn, mode='linear')
    scm.fit(dm)
    return scm, graph, vn


def graph_edge_jaccard(g1, g2):
    """Jaccard similarity between two causal-graph edge sets -- a per-refit
    stability metric, complementing the rolling-window analysis in Section 5.4."""
    e1 = set(zip(*np.nonzero(g1)))
    e2 = set(zip(*np.nonzero(g2)))
    if not e1 and not e2:
        return 1.0
    return len(e1 & e2) / len(e1 | e2)

print('OK  Rolling-SCM refit + graph-stability helpers ready')


OK  Rolling-SCM refit + graph-stability helpers ready


## 10. Validation-Based Early Stopping & Training Loop

In [13]:
def evaluate_on_val(agent, val_data, n_episodes=3):
    sharpes = []
    for _ in range(n_episodes):
        env   = TradingEnvironment(val_data)
        state = env.reset()
        done  = False
        while not done:
            action = agent.act(state, eps=0.)
            state, _, done = env.step(action)
        sharpes.append(env.metrics()['sharpe_ratio'])
    return float(np.mean(sharpes))


def train_agent(agent, train_data, val_data, n_episodes=N_EPISODES,
                use_cer=False, patience=30, verbose=False):
    eps, eps_min, eps_decay = 0.8, 0.02, 0.99
    best_val_sharpe = -np.inf
    best_state_dict = None
    no_improve      = 0
    eval_every      = 10

    for ep in range(n_episodes):
        env   = TradingEnvironment(train_data)
        state = env.reset()
        done  = False

        while not done:
            action = agent.act(state, eps)
            ns, reward, done = env.step(action)
            agent.push(state, action, reward, ns, float(done))

            if use_cer and hasattr(agent, 'generate_and_store_counterfactuals'):
                agent.generate_and_store_counterfactuals(state, action, reward, ns, done)

            agent.train_step()
            state = ns

        eps = max(eps_min, eps * eps_decay)

        # FIX: target sync every TARGET_UPDATE_FREQ episodes, NOT every step
        if hasattr(agent, 'sync_target') and (ep + 1) % TARGET_UPDATE_FREQ == 0:
            agent.sync_target()

        # Validation check
        if val_data is not None and (ep + 1) % eval_every == 0:
            val_sharpe = evaluate_on_val(agent, val_data)
            if val_sharpe > best_val_sharpe:
                best_val_sharpe = val_sharpe
                if hasattr(agent, 'q_net'):
                    best_state_dict = copy.deepcopy(agent.q_net.state_dict())
                elif hasattr(agent, 'actor'):
                    best_state_dict = copy.deepcopy(agent.actor.state_dict())
                no_improve = 0
            else:
                no_improve += eval_every
                if no_improve >= patience:
                    if verbose:
                        print(f'      Early stop ep {ep+1}  best_val_sharpe={best_val_sharpe:.3f}')
                    break

    if best_state_dict is not None:
        if hasattr(agent, 'q_net'):
            agent.q_net.load_state_dict(best_state_dict)
        elif hasattr(agent, 'actor'):
            agent.actor.load_state_dict(best_state_dict)


def train_agent_rolling_scm(agent, train_data, val_data, n_episodes=N_EPISODES,
                             refit_every=ROLLING_SCM_REFIT_EVERY, patience=30, verbose=False):
    """Same loop as train_agent, but periodically re-estimates and hot-swaps the
    agent's linear SCM using a rolling window of train_data, instead of the single
    fixed fit used by 'causal_dqn' (Section 7.7 limitation #3: a fixed SCM may go
    stale mid-deployment). Returns a small stability report for reporting alongside
    the rolling-window graph-stability analysis in Section 5.4."""
    eps, eps_min, eps_decay = 0.8, 0.02, 0.99
    best_val_sharpe = -np.inf
    best_state_dict = None
    no_improve      = 0
    eval_every      = 10
    jaccards        = []
    prev_graph      = None
    n_refits        = 0

    for ep in range(n_episodes):
        if ep % refit_every == 0:
            new_scm, new_graph, _ = refit_linear_scm_rolling(train_data)
            if prev_graph is not None:
                jaccards.append(graph_edge_jaccard(prev_graph, new_graph))
            prev_graph = new_graph
            agent.scm  = new_scm     # hot-swap -- CERMixin reads self.scm at CF-generation time
            n_refits  += 1

        env   = TradingEnvironment(train_data)
        state = env.reset()
        done  = False
        while not done:
            action = agent.act(state, eps)
            ns, reward, done = env.step(action)
            agent.push(state, action, reward, ns, float(done))
            if hasattr(agent, 'generate_and_store_counterfactuals'):
                agent.generate_and_store_counterfactuals(state, action, reward, ns, done)
            agent.train_step()
            state = ns

        eps = max(eps_min, eps * eps_decay)
        if hasattr(agent, 'sync_target') and (ep + 1) % TARGET_UPDATE_FREQ == 0:
            agent.sync_target()

        if val_data is not None and (ep + 1) % eval_every == 0:
            val_sharpe = evaluate_on_val(agent, val_data)
            if val_sharpe > best_val_sharpe:
                best_val_sharpe = val_sharpe
                best_state_dict = copy.deepcopy(agent.q_net.state_dict())
                no_improve = 0
            else:
                no_improve += eval_every
                if no_improve >= patience:
                    if verbose:
                        print(f'      Early stop ep {ep+1}  best_val_sharpe={best_val_sharpe:.3f}')
                    break

    if best_state_dict is not None:
        agent.q_net.load_state_dict(best_state_dict)

    return {'n_refits': n_refits,
            'mean_edge_jaccard': float(np.mean(jaccards)) if jaccards else None}


def train_ppo(agent, train_data, val_data, n_episodes=N_EPISODES, patience=30,
              eval_every=10, verbose=False):
    """On-policy training loop for PPO: collect one full episode of rollout, run
    clipped-surrogate updates, clear the buffer. No replay buffer, no CER mixin
    (Section 7.7 limitation #1)."""
    best_val_sharpe = -np.inf
    best_state_dict = None
    no_improve      = 0

    for ep in range(n_episodes):
        env   = TradingEnvironment(train_data)
        state = env.reset()
        done  = False
        while not done:
            action, log_prob, value = agent.act_train(state)
            ns, reward, done = env.step(action)
            agent.store(state, action, log_prob, reward, float(done), value)
            state = ns

        with torch.no_grad():
            s_last = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(agent.device)
            _, last_value = agent.net(s_last)
        agent.update(last_value=float(last_value.item()))

        if val_data is not None and (ep + 1) % eval_every == 0:
            val_sharpe = evaluate_on_val(agent, val_data)
            if val_sharpe > best_val_sharpe:
                best_val_sharpe = val_sharpe
                best_state_dict = copy.deepcopy(agent.net.state_dict())
                no_improve = 0
            else:
                no_improve += eval_every
                if no_improve >= patience:
                    if verbose:
                        print(f'      Early stop ep {ep+1}  best_val_sharpe={best_val_sharpe:.3f}')
                    break

    if best_state_dict is not None:
        agent.net.load_state_dict(best_state_dict)


def evaluate_agent(agent, test_data, tc=None):
    env   = TradingEnvironment(test_data, tc=TRANS_COST if tc is None else tc)
    state = env.reset()
    done  = False
    while not done:
        action = agent.act(state, eps=0.)
        state, _, done = env.step(action)
    return env.metrics()


def evaluate_agent_multi_tc(agent, test_data, tc_levels=TC_SWEEP_LEVELS):
    """Re-run the SAME frozen policy under several transaction-cost assumptions
    (Section 7.7 limitation #5). Cheap: in this environment the state fed back to
    the agent depends only on price/feature data, not on tc, so action selection is
    unaffected by cost -- this re-scores the same trade sequence, it does not
    retrain. Report it as a policy-fixed cost-sensitivity check, not cost-aware
    retraining."""
    out = {}
    for tc in tc_levels:
        out[f'{int(round(tc * 10000))}bps'] = evaluate_agent(agent, test_data, tc=tc)
    return out


def buy_and_hold(data):
    p   = data['Close'].values
    sh  = INITIAL_BAL / (p[0] * (1 + TRANS_COST))
    fv  = sh * p[-1] * (1 - TRANS_COST)
    r   = pd.Series(p).pct_change().dropna().values
    sh2 = np.mean(r) / (np.std(r) + 1e-9) * np.sqrt(252)
    neg = r[r < 0]
    so  = np.mean(r) / (np.std(neg) + 1e-9) * np.sqrt(252) if len(neg) > 0 else sh2
    cum = (1 + pd.Series(r)).cumprod()
    mdd = float((cum / cum.cummax() - 1).min())
    ann = ((fv / INITIAL_BAL) ** (252 / max(len(r), 1))) - 1
    return {'total_return': (fv - INITIAL_BAL) / INITIAL_BAL,
            'sharpe_ratio': sh2, 'sortino_ratio': so,
            'calmar_ratio': ann / (abs(mdd) + 1e-9),
            'max_drawdown': mdd, 'win_rate': float(np.mean(r > 0)),
            'final_value': fv}


def rule_strategy(data, rule='momentum'):
    env   = TradingEnvironment(data)
    state = env.reset()
    for i in range(len(data) - 1):
        if rule == 'momentum':
            m      = float(data.iloc[i]['momentum_10'])
            action = 2 if m > 0.02 else (0 if m < -0.02 else 1)
        else:
            rsi    = float(data.iloc[i]['rsi'])
            action = 2 if rsi < 0.30 else (0 if rsi > 0.70 else 1)
        state, _, done = env.step(action)
        if done: break
    return env.metrics()

print('OK  Training loop + early stopping + baselines ready (+ rolling-SCM + PPO + tc-sweep)')


OK  Training loop + early stopping + baselines ready (+ rolling-SCM + PPO + tc-sweep)


## 11. Checkpointing Utilities

In [14]:
METHODS_RL     = ['causal_dqn', 'causal_dqn_neural', 'causal_sac', 'causal_sac_linear',
                  'causal_dqn_rolling',
                  'dueling_dqn', 'discrete_sac', 'ppo',
                  'random_cf', 'no_cql', 'no_per']
# causal_dqn          : linear SCM  + DQN   (original "ours")
# causal_dqn_neural   : neural SCM + DQN   (2x2 cell -- isolates SCM-type effect for DQN)
# causal_sac_linear   : linear SCM + SAC   (2x2 cell -- isolates SCM-type effect for SAC)
# causal_sac          : neural SCM + SAC   (original "ours")
# causal_dqn_rolling  : linear SCM, periodically re-estimated on a rolling window during
#                       training, instead of fit once and frozen (Section 7.7 limitation #3)
# ppo                 : on-policy backbone baseline, no CER (Section 7.7 limitation #1)
METHODS_STATIC = ['buy_hold', 'momentum', 'mean_reversion']
ALL_METHODS    = METHODS_RL + METHODS_STATIC

def ckpt_path(ticker):
    return os.path.join(CKPT_DIR, f'{ticker.replace(".", "_")}.json')

def save_ckpt(ticker, data):
    with open(ckpt_path(ticker), 'w') as f:
        json.dump(data, f, indent=2)

def load_ckpt(ticker):
    p = ckpt_path(ticker)
    if os.path.exists(p):
        with open(p) as f:
            return json.load(f)
    return None

def missing_rl_methods(ticker):
    """Which RL methods still need trials for this ticker, plus whether the
    per-asset SCM-fit report and the transaction-cost sweep still need to be
    computed. Handles resuming after METHODS_RL was extended (old checkpoints
    simply have the new keys empty/missing)."""
    ck = load_ckpt(ticker)
    if ck is None:
        return list(METHODS_RL), True, True
    missing        = [m for m in METHODS_RL if len(ck.get(m, [])) < N_TRIALS]
    needs_scm_fit  = 'scm_fit'  not in ck
    needs_tc_sweep = 'tc_sweep' not in ck
    return missing, needs_scm_fit, needs_tc_sweep

def already_done(ticker):
    missing, needs_scm_fit, needs_tc_sweep = missing_rl_methods(ticker)
    return len(missing) == 0 and not needs_scm_fit and not needs_tc_sweep

existing = [t for t in STOCK_TICKERS if already_done(t)]
print(f'OK  Checkpointing ready  |  dir: {CKPT_DIR}/')
print(f'    Already completed: {existing if existing else "none"}')
print(f'    METHODS_RL ({len(METHODS_RL)}): {METHODS_RL}')


OK  Checkpointing ready  |  dir: journal_checkpoints/
    Already completed: ['AAPL', 'MSFT']
    METHODS_RL (11): ['causal_dqn', 'causal_dqn_neural', 'causal_sac', 'causal_sac_linear', 'causal_dqn_rolling', 'dueling_dqn', 'discrete_sac', 'ppo', 'random_cf', 'no_cql', 'no_per']


In [15]:
!nvidia-smi

Thu Jul 30 09:17:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.173.02             Driver Version: 580.173.02     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        Off |   00000000:01:00.0  On |                  N/A |
|  0%   36C    P8             23W /  370W |     695MiB /  24576MiB |      9%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 12. Main Experiment Loop -- 30 Assets x 20 Trials

> **Expected runtime on RTX 3090:** ~4-6 hours
> Checkpoints saved per ticker -- safe to interrupt and resume.


In [16]:
results          = {}
experiment_start = time.time()

def make_agent(method, scm_linear, scm_neural, vn):
    """Factory for every RL variant. Returns (agent, use_cer)."""
    if method == 'causal_dqn':          return CausalDQNAgent(scm_linear, vn), True
    if method == 'causal_dqn_neural':   return CausalDQNAgent(scm_neural, vn), True   # 2x2 cell
    if method == 'causal_sac':          return CausalSACAgent(scm_neural, vn), True
    if method == 'causal_sac_linear':   return CausalSACAgent(scm_linear, vn), True   # 2x2 cell
    if method == 'causal_dqn_rolling':  return CausalDQNAgent(scm_linear, vn), True   # SCM hot-swapped during training
    if method == 'dueling_dqn':         return DuelingDQNAgent(), False
    if method == 'discrete_sac':        return DiscreteSACAgent(), False
    if method == 'ppo':                 return PPOAgent(), False
    if method == 'random_cf':           return RandomCFDQNAgent(), True
    if method == 'no_cql':              return NoCQLDQNAgent(scm_linear, vn), True
    if method == 'no_per':              return NoPERDQNAgent(scm_linear, vn), True
    raise ValueError(method)

for ticker in STOCK_TICKERS:
    print(f'\n{"="*65}')
    print(f'  {ticker}  [{STOCK_TICKERS.index(ticker)+1}/{len(STOCK_TICKERS)}]')
    print(f'{"="*65}')

    needs_rl, needs_scm_fit, needs_tc_sweep = missing_rl_methods(ticker)
    ck  = load_ckpt(ticker)
    res = ck if ck is not None else {}
    for m in METHODS_RL:
        res.setdefault(m, [])

    if not needs_rl and not needs_scm_fit and not needs_tc_sweep:
        print('  OK  Already complete -- loading checkpoint')
        results[ticker] = res
        continue

    print(f'  Resuming: missing methods = {needs_rl if needs_rl else "none"}  |  '
          f'scm_fit needed = {needs_scm_fit}  |  tc_sweep needed = {needs_tc_sweep}')

    # Load data
    train_data = TradingDataLoader(ticker, TRAIN_START, TRAIN_END).load()
    val_data   = TradingDataLoader(ticker, VAL_START,   VAL_END).load()
    test_data  = TradingDataLoader(ticker, TEST_START,  TEST_END).load()

    if train_data is None or test_data is None:
        print('  Skipping -- data unavailable')
        continue
    if val_data is None:
        val_data = train_data.iloc[-int(len(train_data) * 0.15):]
        print(f'  Val fallback: last 15% of train ({len(val_data)} rows)')

    print(f'  Train:{len(train_data)}  Val:{len(val_data)}  Test:{len(test_data)}')

    # Causal structure -- needed whenever any causal_* method is missing, or scm_fit
    # itself hasn't been logged yet for this ticker.
    print('  [1/5] PC causal graph...')
    t_cg = time.time()
    cg_pc, vn, dm = learn_causal_structure(train_data, method='pc')
    print(f'        PC done in {time.time()-t_cg:.1f}s')

    print('  [2/5] NOTEARS-MLP causal graph...')
    t_nt = time.time()
    cg_notears, _, _ = learn_causal_structure(train_data, method='notears')
    print(f'        NOTEARS done in {time.time()-t_nt:.1f}s')

    print('  [3/5] Fitting SCMs...')
    scm_linear = StructuralCausalModel(cg_pc,      vn, mode='linear')
    scm_linear.fit(dm)
    scm_neural = StructuralCausalModel(cg_notears, vn, mode='neural')
    scm_neural.fit(dm)
    rep  = scm_linear.fit_report()
    rep2 = scm_neural.fit_report()
    if rep:
        print(f'    Linear SCM  R2 mean={rep["mean_r2"]:.3f}  min={rep["min_r2"]:.3f}')
    if rep2:
        print(f'    Neural SCM  R2 mean={rep2["mean_r2"]:.3f}  min={rep2["min_r2"]:.3f}')

    # Log per-asset SCM fit quality -- needed for the SCM-fit vs performance-gain
    # correlation analysis (Section 15b).
    res['scm_fit'] = {
        'linear_mean_r2': rep.get('mean_r2')  if rep  else None,
        'linear_min_r2':  rep.get('min_r2')   if rep  else None,
        'linear_per_var': rep.get('per_var')  if rep  else None,
        'neural_mean_r2': rep2.get('mean_r2') if rep2 else None,
        'neural_min_r2':  rep2.get('min_r2')  if rep2 else None,
        'neural_per_var': rep2.get('per_var') if rep2 else None,
    }

    # Deterministic baselines -- only compute if missing
    print('  [4/5] Static baselines...')
    if not res.get('buy_hold'):
        res['buy_hold']       = buy_and_hold(test_data)
        res['momentum']       = rule_strategy(test_data, 'momentum')
        res['mean_reversion'] = rule_strategy(test_data, 'mean_reversion')
    print(f'    B&H {res["buy_hold"]["total_return"]*100:+.1f}%  '
          f'Mom {res["momentum"]["total_return"]*100:+.1f}%  '
          f'MR {res["mean_reversion"]["total_return"]*100:+.1f}%')

    # RL Trials -- only run the (method, trial) pairs still missing. The transaction-
    # cost sweep (Section 7.7 limitation #5) piggybacks on trial 0 of every method,
    # since it needs a live trained agent object (not persisted to disk) and is cheap
    # relative to training (inference-only re-scoring under different tc).
    if needs_rl or needs_tc_sweep:
        active_methods = needs_rl if needs_rl else [m for m in METHODS_RL if m not in needs_rl]
        print(f'  [5/5] {N_TRIALS} trials x {N_EPISODES} episodes  |  methods: {needs_rl}')
        t0 = time.time()

        for trial in range(N_TRIALS):
            set_seed(100 + trial)
            for m in METHODS_RL:
                need_trial   = m in needs_rl and len(res[m]) <= trial
                need_sweep   = (trial == 0) and needs_tc_sweep and m not in res.get('tc_sweep', {})
                if not need_trial and not need_sweep:
                    continue

                ag, use_cer = make_agent(m, scm_linear, scm_neural, vn)

                if m == 'ppo':
                    train_ppo(ag, train_data, val_data)
                elif m == 'causal_dqn_rolling':
                    stab = train_agent_rolling_scm(ag, train_data, val_data)
                    if need_trial:
                        res.setdefault('rolling_scm_stability', []).append(stab)
                else:
                    train_agent(ag, train_data, val_data, use_cer=use_cer)

                if need_trial:
                    res[m].append(evaluate_agent(ag, test_data))
                if need_sweep:
                    res.setdefault('tc_sweep', {})[m] = evaluate_agent_multi_tc(ag, test_data)

                del ag; gc.collect(); torch.cuda.empty_cache()

            if (trial + 1) % 5 == 0:
                elapsed = (time.time() - t0) / 60
                snippet = '  '.join(
                    f'{m} {np.mean([r["total_return"] for r in res[m]])*100:+.1f}%'
                    for m in needs_rl if res[m]
                )
                print(f'    Trial {trial+1:2d}/{N_TRIALS} ({elapsed:.0f}min) | {snippet}')

        print(f'  OK  {ticker} RL trials done in {(time.time()-t0)/60:.1f} min')
    else:
        print('  [5/5] All RL methods + tc_sweep already complete for this ticker -- skipped')

    results[ticker] = res
    save_ckpt(ticker, res)
    print(f'  OK  {ticker} checkpoint saved')

    del train_data, val_data, test_data, cg_pc, cg_notears, dm, scm_linear, scm_neural, rep, rep2
    gc.collect()

total_h = (time.time() - experiment_start) / 3600
print(f'\nOK  All experiments complete in {total_h:.2f} hours')



  AAPL  [1/30]
  OK  Already complete -- loading checkpoint

  MSFT  [2/30]
  OK  Already complete -- loading checkpoint

  NVDA  [3/30]
  Resuming: missing methods = ['causal_dqn', 'causal_dqn_neural', 'causal_sac', 'causal_sac_linear', 'causal_dqn_rolling', 'dueling_dqn', 'discrete_sac', 'ppo', 'random_cf', 'no_cql', 'no_per']  |  scm_fit needed = True  |  tc_sweep needed = True


2026-07-30 09:18:02,464 - /home/nmit/anaconda3/lib/python3.12/site-packages/castle/algorithms/gradient/notears/torch/nonlinear.py[line:135] - INFO: GPU is available.
2026-07-30 09:18:02,479 - /home/nmit/anaconda3/lib/python3.12/site-packages/castle/algorithms/gradient/notears/torch/nonlinear.py[line:237] - INFO: [start]: n=957, d=12, iter_=100, h_=1e-08, rho_=1e+16


  Train:958  Val:202  Test:452
  [1/5] PC causal graph...
    [PC alpha=0.05] 12 nodes, 16 edges
        PC done in 0.1s
  [2/5] NOTEARS-MLP causal graph...


2026-07-30 09:18:13,596 - /home/nmit/anaconda3/lib/python3.12/site-packages/castle/algorithms/gradient/notears/torch/nonlinear.py[line:249] - INFO: FINISHED


    [NOTEARS-MLP] 12 nodes, 18 edges
        NOTEARS done in 11.1s
  [3/5] Fitting SCMs...
    Linear SCM  R2 mean=0.561  min=0.019
    Neural SCM  R2 mean=0.573  min=0.033
  [4/5] Static baselines...
    B&H +466.7%  Mom +311.4%  MR +112.7%
  [5/5] 10 trials x 100 episodes  |  methods: ['causal_dqn', 'causal_dqn_neural', 'causal_sac', 'causal_sac_linear', 'causal_dqn_rolling', 'dueling_dqn', 'discrete_sac', 'ppo', 'random_cf', 'no_cql', 'no_per']
    [PC alpha=0.05] 12 nodes, 20 edges
    [PC alpha=0.05] 12 nodes, 20 edges
    [PC alpha=0.05] 12 nodes, 20 edges
    [PC alpha=0.05] 12 nodes, 20 edges
    [PC alpha=0.05] 12 nodes, 20 edges
    [PC alpha=0.05] 12 nodes, 20 edges
    [PC alpha=0.05] 12 nodes, 20 edges
    [PC alpha=0.05] 12 nodes, 20 edges
    [PC alpha=0.05] 12 nodes, 20 edges
    [PC alpha=0.05] 12 nodes, 20 edges
    [PC alpha=0.05] 12 nodes, 20 edges
    [PC alpha=0.05] 12 nodes, 20 edges
    [PC alpha=0.05] 12 nodes, 20 edges
    [PC alpha=0.05] 12 nodes, 20 edges
  

...

  WARN  IAU: Only 0 rows -- insufficient history


2026-08-08 13:50:50,643 - /home/nmit/anaconda3/lib/python3.12/site-packages/yfinance/base.py[line:190] - ERROR: Failed to get ticker 'IAU' reason: Failed to perform, curl: (28) Resolving timed out after 10000 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
2026-08-08 13:50:50,644 - /home/nmit/anaconda3/lib/python3.12/site-packages/yfinance/scrapers/history.py[line:139] - ERROR: $IAU: possibly delisted; no timezone found
2026-08-08 13:50:50,645 - /home/nmit/anaconda3/lib/python3.12/site-packages/yfinance/multi.py[line:187] - ERROR: 
1 Failed download:
2026-08-08 13:50:50,645 - /home/nmit/anaconda3/lib/python3.12/site-packages/yfinance/multi.py[line:195] - ERROR: ['IAU']: possibly delisted; no timezone found


  WARN  IAU: Only 0 rows -- insufficient history


2026-08-08 13:51:02,931 - /home/nmit/anaconda3/lib/python3.12/site-packages/yfinance/base.py[line:190] - ERROR: Failed to get ticker 'IAU' reason: Failed to perform, curl: (28) Resolving timed out after 10000 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
2026-08-08 13:51:02,932 - /home/nmit/anaconda3/lib/python3.12/site-packages/yfinance/scrapers/history.py[line:139] - ERROR: $IAU: possibly delisted; no timezone found
2026-08-08 13:51:02,938 - /home/nmit/anaconda3/lib/python3.12/site-packages/yfinance/multi.py[line:187] - ERROR: 
1 Failed download:
2026-08-08 13:51:02,938 - /home/nmit/anaconda3/lib/python3.12/site-packages/yfinance/multi.py[line:195] - ERROR: ['IAU']: possibly delisted; no timezone found


  WARN  IAU: Only 0 rows -- insufficient history
  Skipping -- data unavailable

OK  All experiments complete in 220.55 hours


## 13. Statistical Analysis -- Journal Grade

In [17]:
valid = [t for t in STOCK_TICKERS if t in results and results[t].get('buy_hold')]
print(f'Valid tickers: {len(valid)}')

def flat_metric(method, metric):
    vals = []
    if method in METHODS_RL:
        for t in valid:
            trials = results[t].get(method, [])
            if trials:
                vals.extend([r[metric] for r in trials if metric in r])
    else:
        for t in valid:
            m = results[t].get(method)
            if isinstance(m, dict) and metric in m:
                vals.append(m[metric])
    return vals

all_ret    = {m: flat_metric(m, 'total_return')  for m in ALL_METHODS}
all_sharpe = {m: flat_metric(m, 'sharpe_ratio')  for m in ALL_METHODS}
all_sort   = {m: flat_metric(m, 'sortino_ratio') for m in METHODS_RL}
all_cal    = {m: flat_metric(m, 'calmar_ratio')  for m in METHODS_RL}


# Wilcoxon: CausalDQN vs every other method
cr      = all_ret['causal_dqn']
w_pvals = {}
c_ds    = {}
for m in [x for x in ALL_METHODS if x != 'causal_dqn']:
    comp = all_ret[m]
    n    = min(len(cr), len(comp))
    if n < 10:
        w_pvals[m] = float('nan')
        c_ds[m]    = float('nan')
        continue
    try:
        _, p = stats.wilcoxon(cr[:n], comp[:n])
    except Exception:
        _, p = stats.mannwhitneyu(cr[:n], comp[:n], alternative='two-sided')
    w_pvals[m] = p
    md_ = np.mean(cr) - np.mean(comp)
    ps_ = np.sqrt((np.var(cr) + np.var(comp)) / 2)
    c_ds[m] = md_ / (ps_ + 1e-9)

valid_keys    = [k for k, v in w_pvals.items() if not np.isnan(v)]
_, pvals_corr, _, _ = multipletests([w_pvals[k] for k in valid_keys], method='bonferroni')
corrected = dict(zip(valid_keys, pvals_corr))

# Friedman across RL methods
per_stock = {m: [np.mean([r['total_return'] for r in results[t][m]])
                 for t in valid if results[t].get(m)]
             for m in METHODS_RL}
try:
    f_stat, f_pval = stats.friedmanchisquare(*[per_stock[m] for m in METHODS_RL
                                               if len(per_stock[m]) > 0])
except Exception as e:
    f_stat, f_pval = float('nan'), float('nan')
    print(f'Friedman: {e}')

print('='*82)
print('JOURNAL STATISTICAL RESULTS')
print('='*82)
print(f'{"Method":<22}{"Mean Ret%":>10}{"+-Std":>7}{"Sharpe":>8}'
      f'{"Wilcox p":>12}{"Corr p":>10}{"d":>8}')
print('-'*82)
for m in ALL_METHODS:
    mr  = np.mean(all_ret[m]) * 100   if all_ret[m]    else float('nan')
    sr  = np.std(all_ret[m])  * 100   if all_ret[m]    else float('nan')
    sh  = np.mean(all_sharpe[m])      if all_sharpe[m] else float('nan')
    wp  = w_pvals.get(m, float('nan'))
    cp  = corrected.get(m, float('nan'))
    cd_ = c_ds.get(m, float('nan'))
    sig = '**' if cp < 0.01 else ('*' if cp < 0.05 else '')
    print(f'{m:<22}{mr:>+9.2f}%{sr:>6.2f}%{sh:>8.3f}'
          f'{wp:>11.4f}{cp:>9.4f}{sig:<2}{cd_:>+7.3f}')

print(f'\nFriedman (RL methods): chi2={f_stat:.3f}  p={f_pval:.6f}')
print('* p<0.05  ** p<0.01  (Bonferroni corrected)')


Valid tickers: 24
JOURNAL STATISTICAL RESULTS
Method                 Mean Ret%  +-Std  Sharpe    Wilcox p    Corr p       d
----------------------------------------------------------------------------------
causal_dqn               +22.48% 46.87%   0.446        nan      nan     +nan
causal_dqn_neural        +32.89% 56.46%   0.589     0.0012   0.0162*  -0.201
causal_sac               +35.01% 52.30%   0.731     0.0003   0.0038** -0.252
causal_sac_linear        +33.62% 44.98%   0.724     0.0003   0.0043** -0.242
causal_dqn_rolling       +33.29% 58.70%   0.625     0.0032   0.0414*  -0.203
dueling_dqn              +30.46% 50.51%   0.564     0.0068   0.0882   -0.164
discrete_sac             +24.64% 40.57%   0.524     0.2355   1.0000   -0.049
ppo                      +60.49% 77.86%   0.986     0.0000   0.0000** -0.591
random_cf                +17.36% 30.25%   0.370     0.6438   1.0000   +0.130
no_cql                   +32.77% 51.43%   0.625     0.0002   0.0023** -0.209
no_per                 

## 13b. Sector-Clustered Bootstrap Tests (Section 7.7 limitation #6)
The Wilcoxon test above treats the 30 assets as independent observations, but within-sector correlation means they aren't. This resamples SECTORS (not individual assets) with replacement, so the resulting confidence intervals and p-values respect that clustering instead of assuming it away.


In [18]:
np.random.seed(42)
N_BOOT    = 10000
sector_of = {t: s for s, lst in ASSET_UNIVERSE.items() for t in lst}
sectors   = list(ASSET_UNIVERSE.keys())

def _per_asset_mean_return(method):
    return {t: np.mean([r['total_return'] for r in results[t][method]])
            for t in valid if results[t].get(method)}

def cluster_bootstrap_diff(method_a, method_b, n_boot=N_BOOT):
    """Sector-level cluster bootstrap for the paired per-asset mean-return
    difference (method_a - method_b)."""
    per_asset_a = _per_asset_mean_return(method_a)
    per_asset_b = _per_asset_mean_return(method_b)
    common = [t for t in per_asset_a if t in per_asset_b]
    if len(common) < 5:
        return None

    by_sector   = {s: [t for t in common if sector_of[t] == s] for s in sectors}
    by_sector   = {s: v for s, v in by_sector.items() if v}
    sector_list = list(by_sector.keys())

    observed = np.mean([per_asset_a[t] - per_asset_b[t] for t in common])

    boot_diffs = np.empty(n_boot)
    for b in range(n_boot):
        chosen = np.random.choice(sector_list, size=len(sector_list), replace=True)
        pooled = [t for s in chosen for t in by_sector[s]]
        boot_diffs[b] = np.mean([per_asset_a[t] - per_asset_b[t] for t in pooled])

    ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])
    p_two_sided  = 2 * min((boot_diffs <= 0).mean(), (boot_diffs >= 0).mean())
    return {'n_assets': len(common), 'n_sectors': len(sector_list),
            'observed_diff': observed, 'ci_lo': ci_lo, 'ci_hi': ci_hi,
            'boot_p': min(p_two_sided, 1.0)}

comparisons = [('causal_dqn', 'dueling_dqn'), ('causal_sac', 'discrete_sac'),
               ('causal_dqn', 'random_cf'), ('causal_sac', 'ppo')]
cluster_rows = []
for a, b in comparisons:
    r = cluster_bootstrap_diff(a, b)
    if r is None:
        continue
    cluster_rows.append({'Comparison': f'{a} vs {b}', **r})
    print(f'{a} vs {b}: diff={r["observed_diff"]*100:+.2f}%  '
          f'95% cluster-bootstrap CI=[{r["ci_lo"]*100:+.2f}%, {r["ci_hi"]*100:+.2f}%]  '
          f'p={r["boot_p"]:.4f}  (n={r["n_assets"]} assets, {r["n_sectors"]} sectors)')

pd.DataFrame(cluster_rows).to_csv('cluster_robust_bootstrap.csv', index=False)
print('\nOK  Saved: cluster_robust_bootstrap.csv')
print('Report these sector-clustered CIs/p-values alongside (not instead of) the asset-level')
print('Wilcoxon test above -- they directly address the independence assumption.')


causal_dqn vs dueling_dqn: diff=-7.98%  95% cluster-bootstrap CI=[-21.23%, +2.06%]  p=0.2414  (n=24 assets, 4 sectors)
causal_sac vs discrete_sac: diff=+10.38%  95% cluster-bootstrap CI=[+2.46%, +24.52%]  p=0.0000  (n=24 assets, 4 sectors)
causal_dqn vs random_cf: diff=+5.12%  95% cluster-bootstrap CI=[-5.42%, +18.69%]  p=0.4114  (n=24 assets, 4 sectors)
causal_sac vs ppo: diff=-25.48%  95% cluster-bootstrap CI=[-47.90%, -10.15%]  p=0.0000  (n=24 assets, 4 sectors)

OK  Saved: cluster_robust_bootstrap.csv
Report these sector-clustered CIs/p-values alongside (not instead of) the asset-level
Wilcoxon test above -- they directly address the independence assumption.


## 13c. Bootstrap Effect-Size CIs & Formal Power Analysis (Section 7.7 limitation #9)
Formalizes the manuscript's post-hoc power statement with code: bootstrap CIs on Cohen's d (using the same sector-cluster resampling as above) plus the standard achieved-power and required-n calculations at the Bonferroni-corrected alpha.


In [19]:
from statsmodels.stats.power import TTestIndPower

def bootstrap_cohens_d_ci(method_a, method_b, n_boot=N_BOOT):
    per_asset_a = _per_asset_mean_return(method_a)
    per_asset_b = _per_asset_mean_return(method_b)
    common = [t for t in per_asset_a if t in per_asset_b]
    if len(common) < 5:
        return None
    by_sector   = {s: [t for t in common if sector_of[t] == s] for s in sectors}
    by_sector   = {s: v for s, v in by_sector.items() if v}
    sector_list = list(by_sector.keys())

    def cohend(ts):
        xa = np.array([per_asset_a[t] for t in ts])
        xb = np.array([per_asset_b[t] for t in ts])
        pooled_sd = np.sqrt((xa.var() + xb.var()) / 2)
        return (xa.mean() - xb.mean()) / (pooled_sd + 1e-9)

    observed_d = cohend(common)
    boot_d = np.empty(n_boot)
    for b in range(n_boot):
        chosen = np.random.choice(sector_list, size=len(sector_list), replace=True)
        pooled = [t for s in chosen for t in by_sector[s]]
        boot_d[b] = cohend(pooled)

    ci_lo, ci_hi = np.percentile(boot_d, [2.5, 97.5])
    return observed_d, ci_lo, ci_hi

alpha_bonf = 0.05 / (len(ALL_METHODS) - 1)   # matches the manuscript's Bonferroni correction
power_rows = []
for a, b in comparisons:
    d_result = bootstrap_cohens_d_ci(a, b)
    if d_result is None:
        continue
    d_obs, d_lo, d_hi = d_result
    analysis = TTestIndPower()
    achieved_power = analysis.power(effect_size=abs(d_obs), nobs1=len(valid),
                                     alpha=alpha_bonf, ratio=1.0)
    n_for_80pct = analysis.solve_power(effect_size=abs(d_obs), power=0.80,
                                        alpha=alpha_bonf, ratio=1.0)
    power_rows.append({
        'Comparison': f'{a} vs {b}', 'Cohens_d': d_obs,
        'd_CI_lo': d_lo, 'd_CI_hi': d_hi,
        'achieved_power_n30': achieved_power,
        'n_required_80pct_power': n_for_80pct,
    })
    print(f'{a} vs {b}: d={d_obs:+.3f} (95% CI [{d_lo:+.3f}, {d_hi:+.3f}])  '
          f'achieved power (n={len(valid)})={achieved_power*100:.1f}%  '
          f'n needed for 80% power={n_for_80pct:.0f}')

print('\nNote: the power/required-n figures use the standard independent-samples formula')
print('(statsmodels), matching how the manuscript already reports post-hoc power. The')
print('cluster-bootstrap CI on d is a separate, complementary check on whether the point')
print('estimate itself is reliable given sector correlation -- the two are not the same')
print('correction stacked twice.')

pd.DataFrame(power_rows).to_csv('power_analysis.csv', index=False)
print('OK  Saved: power_analysis.csv')


causal_dqn vs dueling_dqn: d=-0.263 (95% CI [-0.521, +0.102])  achieved power (n=24)=2.2%  n needed for 80% power=406
causal_sac vs discrete_sac: d=+0.353 (95% CI [+0.111, +0.624])  achieved power (n=24)=4.3%  n needed for 80% power=226
causal_dqn vs random_cf: d=+0.212 (95% CI [-0.238, +0.756])  achieved power (n=24)=1.5%  n needed for 80% power=621
causal_sac vs ppo: d=-0.466 (95% CI [-0.859, -0.405])  achieved power (n=24)=8.9%  n needed for 80% power=130

Note: the power/required-n figures use the standard independent-samples formula
(statsmodels), matching how the manuscript already reports post-hoc power. The
cluster-bootstrap CI on d is a separate, complementary check on whether the point
estimate itself is reliable given sector correlation -- the two are not the same
correction stacked twice.
OK  Saved: power_analysis.csv


## 14. Ablation Study Table

In [20]:
ABLATION_LABELS = {
    'causal_dqn'         : 'Full CER-DQN (ours): linear SCM + DQN',
    'causal_dqn_neural'  : 'CER-DQN, neural SCM (2x2 cell)',
    'causal_sac_linear'  : 'CER-SAC, linear SCM (2x2 cell)',
    'causal_sac'         : 'Full CER-SAC (ours): neural SCM + SAC',
    'causal_dqn_rolling' : 'CER-DQN, rolling SCM re-estimation (robustness check)',
    'dueling_dqn'        : '-CER (Dueling DQN only)',
    'ppo'                : 'PPO (on-policy backbone, no CER)',
    'random_cf'          : '-SCM (random perturbation)',
    'no_cql'             : '-CQL (no conservative penalty)',
    'no_per'             : '-PER (uniform replay)',
}

abl_rows = []
for m, label in ABLATION_LABELS.items():
    rets = flat_metric(m, 'total_return')
    sh   = flat_metric(m, 'sharpe_ratio')
    so   = flat_metric(m, 'sortino_ratio')
    ca   = flat_metric(m, 'calmar_ratio')
    mdd  = flat_metric(m, 'max_drawdown')
    wr   = flat_metric(m, 'win_rate')
    abl_rows.append({
        'Variant' : label,
        'Ret%'    : f'{np.mean(rets)*100:+.2f}+-{np.std(rets)*100:.2f}' if rets else 'N/A',
        'Sharpe'  : f'{np.mean(sh):.3f}'  if sh  else 'N/A',
        'Sortino' : f'{np.mean(so):.3f}'  if so  else 'N/A',
        'Calmar'  : f'{np.mean(ca):.3f}'  if ca  else 'N/A',
        'MaxDD%'  : f'{np.mean(mdd)*100:.2f}' if mdd else 'N/A',
        'WinRate' : f'{np.mean(wr):.3f}'  if wr  else 'N/A',
    })

df_abl = pd.DataFrame(abl_rows)
print('ABLATION STUDY')
print(df_abl.to_string(index=False))
df_abl.to_csv('journal_ablation.csv', index=False)
print('\nOK  Saved: journal_ablation.csv')


ABLATION STUDY
                                              Variant          Ret% Sharpe   Sortino Calmar MaxDD% WinRate
                Full CER-DQN (ours): linear SCM + DQN +22.48+-46.87  0.446     0.644  0.728 -14.93   0.214
                       CER-DQN, neural SCM (2x2 cell) +32.89+-56.46  0.589  -115.386  0.986 -14.56   0.255
                       CER-SAC, linear SCM (2x2 cell) +33.62+-44.98  0.724  7435.682  1.266 -19.40   0.268
                Full CER-SAC (ours): neural SCM + SAC +35.01+-52.30  0.731  3049.232  1.284 -16.89   0.260
CER-DQN, rolling SCM re-estimation (robustness check) +33.29+-58.70  0.625  3512.376  1.088 -15.50   0.253
                              -CER (Dueling DQN only) +30.46+-50.51  0.564   600.776  1.022 -14.76   0.235
                     PPO (on-policy backbone, no CER) +60.49+-77.86  0.986  3236.711  1.809 -20.84   0.386
                           -SCM (random perturbation) +17.36+-30.25  0.370  4806.244  0.587  -8.55   0.168
                      

## 14b. 2x2 Factorial -- SCM Type (linear/neural) x Algorithm (DQN/SAC)
Isolates whether the CausalDQN-vs-CausalSAC gap in Section 7.4 reflects the SCM used or the RL algorithm used, by running every (SCM, algorithm) combination rather than only the two pairings used for the headline result.


In [21]:
factorial_cells = {
    ('linear', 'DQN'): 'causal_dqn',
    ('neural', 'DQN'): 'causal_dqn_neural',
    ('linear', 'SAC'): 'causal_sac_linear',
    ('neural', 'SAC'): 'causal_sac',
}

fac_rows = []
for (scm_type, algo), m in factorial_cells.items():
    rets = flat_metric(m, 'total_return')
    sh   = flat_metric(m, 'sharpe_ratio')
    fac_rows.append({
        'SCM': scm_type, 'Algorithm': algo, 'Method': m,
        'MeanRet%': np.mean(rets) * 100 if rets else float('nan'),
        'StdRet%':  np.std(rets)  * 100 if rets else float('nan'),
        'Sharpe':   np.mean(sh)         if sh   else float('nan'),
        'n':        len(rets),
    })

df_fac = pd.DataFrame(fac_rows)
print('2x2 FACTORIAL -- MEAN RETURN (%)')
pivot_ret = df_fac.pivot(index='SCM', columns='Algorithm', values='MeanRet%')
print(pivot_ret.to_string())

print('\n2x2 FACTORIAL -- SHARPE')
pivot_sh = df_fac.pivot(index='SCM', columns='Algorithm', values='Sharpe')
print(pivot_sh.to_string())

# Simple main-effects / interaction decomposition (row/col/grand means on returns)
grand = pivot_ret.values.mean()
scm_main_effect  = pivot_ret.mean(axis=1) - grand      # linear vs neural, averaged over algo
algo_main_effect = pivot_ret.mean(axis=0) - grand      # DQN vs SAC, averaged over SCM
interaction = pivot_ret - grand
for s in pivot_ret.index:
    for a in pivot_ret.columns:
        interaction.loc[s, a] -= (scm_main_effect[s] + algo_main_effect[a])

print(f'\nSCM main effect (Ret% - grand mean):\n{scm_main_effect.to_string()}')
print(f'\nAlgorithm main effect (Ret% - grand mean):\n{algo_main_effect.to_string()}')
print(f'\nInteraction term (SCM x Algorithm), Ret% points:\n{interaction.to_string()}')
print('\nInterpretation: if |interaction| is small relative to the main effects, the '
      'CausalDQN-vs-CausalSAC gap in the headline comparison is mostly explained by the '
      'algorithm (or mostly by the SCM), not a synergy specific to the pairing used in Section 7.4.')

df_fac.to_csv('journal_2x2_factorial.csv', index=False)
print('\nOK  Saved: journal_2x2_factorial.csv')


2x2 FACTORIAL -- MEAN RETURN (%)
Algorithm        DQN        SAC
SCM                            
linear     22.479873  33.618402
neural     32.888055  35.013458

2x2 FACTORIAL -- SHARPE
Algorithm       DQN       SAC
SCM                          
linear     0.446401  0.723852
neural     0.589138  0.730935

SCM main effect (Ret% - grand mean):
SCM
linear   -2.950809
neural    2.950809

Algorithm main effect (Ret% - grand mean):
Algorithm
DQN   -3.315983
SAC    3.315983

Interaction term (SCM x Algorithm), Ret% points:
Algorithm       DQN       SAC
SCM                          
linear    -2.253282  2.253282
neural     2.253282 -2.253282

Interpretation: if |interaction| is small relative to the main effects, the CausalDQN-vs-CausalSAC gap in the headline comparison is mostly explained by the algorithm (or mostly by the SCM), not a synergy specific to the pairing used in Section 7.4.

OK  Saved: journal_2x2_factorial.csv


## 15. Per-Asset Results Table

In [22]:
rows = []
for t in valid:
    sector = next((s for s, lst in ASSET_UNIVERSE.items() if t in lst), 'Unknown')
    row    = {'Asset': t, 'Sector': sector}
    scm_fit = results[t].get('scm_fit', {}) or {}
    row['linear_scm_r2'] = scm_fit.get('linear_mean_r2')
    row['neural_scm_r2'] = scm_fit.get('neural_mean_r2')
    for m in METHODS_RL + METHODS_STATIC:
        if m in METHODS_RL and results[t].get(m):
            row[f'{m}_ret%']   = f'{np.mean([r["total_return"] for r in results[t][m]])*100:+.2f}'
            row[f'{m}_sharpe'] = f'{np.mean([r["sharpe_ratio"] for r in results[t][m]]):.3f}'
        elif m in METHODS_STATIC and isinstance(results[t].get(m), dict):
            row[f'{m}_ret%'] = f'{results[t][m]["total_return"]*100:+.2f}'
    rows.append(row)

df_full = pd.DataFrame(rows)
display_cols = ['Asset', 'Sector', 'linear_scm_r2', 'neural_scm_r2',
                'causal_dqn_ret%', 'causal_dqn_sharpe',
                'causal_dqn_neural_ret%',
                'causal_sac_linear_ret%',
                'causal_sac_ret%',
                'dueling_dqn_ret%', 'discrete_sac_ret%',
                'buy_hold_ret%']
print(df_full[[c for c in display_cols if c in df_full.columns]].to_string(index=False))
df_full.to_csv('journal_full_results.csv', index=False)
print('\nOK  Saved: journal_full_results.csv')


    Asset     Sector  linear_scm_r2  neural_scm_r2 causal_dqn_ret% causal_dqn_sharpe causal_dqn_neural_ret% causal_sac_linear_ret% causal_sac_ret% dueling_dqn_ret% discrete_sac_ret% buy_hold_ret%
     AAPL    US_Tech       0.457867       0.655709           +1.94             0.045                 +19.65                 +14.22          +28.64           +24.56            +15.26        +65.99
     MSFT    US_Tech       0.512768       0.586132          +17.58             0.409                 +29.75                 +17.60          +25.09           +16.79            +33.71        +61.94
     NVDA    US_Tech       0.561379       0.573217         +104.00             0.774                +204.29                +130.32         +185.62          +165.33            +60.41       +466.67
    GOOGL    US_Tech       0.479554       0.634030           +4.25             0.094                  +3.15                 +20.36          +55.72           +21.88            +24.74        +99.30
     META    US_Tech

## 15b. Asset-Level Correlation -- SCM Fit Quality vs CER Performance Gain
Tests the central hypothesis directly at the resolution it actually matters: does a better-fitting structural equation model for an asset predict a bigger CER benefit for that asset? Uses the per-asset R^2 already logged in `scm_fit` and the per-asset results already collected in `results`.


In [23]:
from scipy.stats import pearsonr, spearmanr

def asset_level_gain_vs_fit(causal_method, baseline_method, r2_key):
    """Per-asset: (SCM R^2, CER gain over matched baseline)."""
    xs, ys, labels = [], [], []
    for t in valid:
        scm_fit = results[t].get('scm_fit', {}) or {}
        r2 = scm_fit.get(r2_key)
        cm = results[t].get(causal_method, [])
        bm = results[t].get(baseline_method, [])
        if r2 is None or not cm or not bm:
            continue
        gain = np.mean([r['total_return'] for r in cm]) - np.mean([r['total_return'] for r in bm])
        xs.append(r2); ys.append(gain); labels.append(t)
    return np.array(xs), np.array(ys), labels

pairs = [
    ('causal_dqn', 'dueling_dqn',   'linear_mean_r2', 'Linear SCM R2  ->  CER-DQN gain over Dueling-DQN'),
    ('causal_sac', 'discrete_sac',  'neural_mean_r2', 'Neural SCM R2  ->  CER-SAC gain over Discrete-SAC'),
]

corr_results = {}
for causal_m, base_m, r2_key, title in pairs:
    xs, ys, labels = asset_level_gain_vs_fit(causal_m, base_m, r2_key)
    if len(xs) < 4:
        print(f'{title}: not enough assets with both R2 and results (n={len(xs)})')
        continue
    r_p, p_p = pearsonr(xs, ys)
    r_s, p_s = spearmanr(xs, ys)
    corr_results[causal_m] = {'n': len(xs), 'pearson_r': r_p, 'pearson_p': p_p,
                              'spearman_r': r_s, 'spearman_p': p_s}
    print(f'{title}')
    print(f'  n={len(xs)}  Pearson r={r_p:+.3f} (p={p_p:.4f})   Spearman rho={r_s:+.3f} (p={p_s:.4f})')

    plt.figure(figsize=(5, 4))
    plt.scatter(xs, ys, s=40)
    for x, y, lab in zip(xs, ys, labels):
        plt.annotate(lab, (x, y), fontsize=7, xytext=(3, 3), textcoords='offset points')
    z = np.polyfit(xs, ys, 1)
    xr = np.linspace(xs.min(), xs.max(), 50)
    plt.plot(xr, np.poly1d(z)(xr), '--', color='gray', linewidth=1)
    plt.xlabel('SCM mean R^2 (structural-equation fit quality)')
    plt.ylabel('CER gain in total return over baseline')
    plt.title(title, fontsize=9)
    plt.tight_layout()
    plt.savefig(f'scm_fit_vs_gain_{causal_m}.pdf')
    plt.savefig(f'scm_fit_vs_gain_{causal_m}.png', dpi=200)
    plt.show()

pd.DataFrame(corr_results).T.to_csv('scm_fit_correlation.csv')
print('\nOK  Saved: scm_fit_correlation.csv, scm_fit_vs_gain_*.pdf/png')


Linear SCM R2  ->  CER-DQN gain over Dueling-DQN
  n=24  Pearson r=-0.156 (p=0.4675)   Spearman rho=-0.036 (p=0.8686)
Neural SCM R2  ->  CER-SAC gain over Discrete-SAC
  n=24  Pearson r=-0.159 (p=0.4569)   Spearman rho=+0.077 (p=0.7223)

OK  Saved: scm_fit_correlation.csv, scm_fit_vs_gain_*.pdf/png


## 15c. Transaction-Cost Sensitivity (Section 7.7 limitation #5)
Re-scores each trained policy's frozen action sequence under several assumed transaction costs (5/10/20/50 bps), holding the policy fixed. This tests whether the CER advantage survives higher, more retail-like costs -- it does NOT retrain under each cost, which would be needed to answer whether the agent would trade differently if it had been trained under higher costs.


In [24]:
tc_rows = []
sweep_methods = ['causal_dqn', 'causal_sac', 'dueling_dqn', 'discrete_sac', 'ppo']
for tc_label in [f'{bp}bps' for bp in TC_SWEEP_BPS]:
    for m in sweep_methods:
        rets = [results[t]['tc_sweep'][m][tc_label]['total_return']
                for t in valid
                if results[t].get('tc_sweep', {}).get(m, {}).get(tc_label)]
        if rets:
            tc_rows.append({'Method': m, 'Cost': tc_label,
                             'MeanRet%': np.mean(rets) * 100, 'n': len(rets)})

df_tc = pd.DataFrame(tc_rows)
if not df_tc.empty:
    pivot_tc  = df_tc.pivot(index='Method', columns='Cost', values='MeanRet%')
    bps_order = [f'{bp}bps' for bp in TC_SWEEP_BPS]
    pivot_tc  = pivot_tc[[c for c in bps_order if c in pivot_tc.columns]]
    print('TRANSACTION-COST SENSITIVITY -- Mean Return (%) by assumed cost')
    print(pivot_tc.to_string())

    if 'causal_dqn' in pivot_tc.index and 'dueling_dqn' in pivot_tc.index:
        gap = pivot_tc.loc['causal_dqn'] - pivot_tc.loc['dueling_dqn']
        print(f'\nCausalDQN - DuelingDQN gap by cost level (pp):\n{gap.to_string()}')
    if 'causal_sac' in pivot_tc.index and 'discrete_sac' in pivot_tc.index:
        gap2 = pivot_tc.loc['causal_sac'] - pivot_tc.loc['discrete_sac']
        print(f'\nCausalSAC - DiscreteSAC gap by cost level (pp):\n{gap2.to_string()}')

    plt.figure(figsize=(6, 4))
    for m in pivot_tc.index:
        plt.plot(bps_order, pivot_tc.loc[m], marker='o', label=m)
    plt.xlabel('Assumed transaction cost'); plt.ylabel('Mean total return (%)')
    plt.title('Return vs assumed transaction cost (policy fixed, trained at 10bps)', fontsize=9)
    plt.legend(fontsize=8); plt.tight_layout()
    plt.savefig('tc_sensitivity.pdf'); plt.savefig('tc_sensitivity.png', dpi=200)
    plt.show()

    df_tc.to_csv('tc_sensitivity.csv', index=False)
    print('\nOK  Saved: tc_sensitivity.csv, tc_sensitivity.pdf/png')
else:
    print('No tc_sweep data found -- re-run the main loop to populate it.')


TRANSACTION-COST SENSITIVITY -- Mean Return (%) by assumed cost
Cost               5bps      10bps      20bps      50bps
Method                                                  
causal_dqn    35.020348  31.813015  17.936497   9.627887
causal_sac    55.143229  52.028717  46.087479  30.327720
discrete_sac   0.276108  -0.155335  -0.999310  -3.390182
dueling_dqn   30.904112  25.704646  13.319386   1.213258
ppo           65.541281  63.248583  58.826452  46.758847

CausalDQN - DuelingDQN gap by cost level (pp):
Cost
5bps     4.116237
10bps    6.108369
20bps    4.617111
50bps    8.414629

CausalSAC - DiscreteSAC gap by cost level (pp):
Cost
5bps     54.867120
10bps    52.184052
20bps    47.086789
50bps    33.717902

OK  Saved: tc_sensitivity.csv, tc_sensitivity.pdf/png


## 15d. Fixed vs. Rolling SCM (Section 7.7 limitation #3)
Compares the paper's fixed-SCM CER-DQN against a variant that periodically re-estimates the linear SCM on a rolling window during training, and reports the graph-edge stability across those re-estimations -- a direct empirical check of whether the fixed-SCM assumption costs anything in practice.


In [25]:
roll_ret = flat_metric('causal_dqn_rolling', 'total_return')
fix_ret  = flat_metric('causal_dqn', 'total_return')

print('FIXED vs ROLLING SCM -- CER-DQN')
if fix_ret:
    print(f'  Fixed SCM   (causal_dqn)        : {np.mean(fix_ret)*100:+.2f}% +- {np.std(fix_ret)*100:.2f}%')
if roll_ret:
    print(f'  Rolling SCM (causal_dqn_rolling): {np.mean(roll_ret)*100:+.2f}% +- {np.std(roll_ret)*100:.2f}%')

if len(fix_ret) >= 10 and len(roll_ret) >= 10:
    n = min(len(fix_ret), len(roll_ret))
    try:
        _, p_roll = stats.wilcoxon(fix_ret[:n], roll_ret[:n])
    except Exception:
        _, p_roll = stats.mannwhitneyu(fix_ret[:n], roll_ret[:n], alternative='two-sided')
    print(f'  Wilcoxon fixed vs rolling: p={p_roll:.4f}')

stab = [s['mean_edge_jaccard'] for t in valid
        for s in results[t].get('rolling_scm_stability', [])
        if s.get('mean_edge_jaccard') is not None]
if stab:
    print(f'\nGraph stability across mid-training refits (mean edge Jaccard, 1.0=identical): '
          f'{np.mean(stab):.3f} +- {np.std(stab):.3f}')
    print('Ties the rolling-window graph-stability analysis in Section 5.4 directly to whether')
    print('re-estimating mid-training changes realized trading performance.')
else:
    print('\nNo rolling_scm_stability data found -- re-run the main loop to populate it.')

pd.DataFrame({'fixed_ret': pd.Series(fix_ret), 'rolling_ret': pd.Series(roll_ret)}).to_csv(
    'fixed_vs_rolling_scm.csv', index=False)
print('OK  Saved: fixed_vs_rolling_scm.csv')


FIXED vs ROLLING SCM -- CER-DQN
  Fixed SCM   (causal_dqn)        : +22.48% +- 46.87%
  Rolling SCM (causal_dqn_rolling): +33.29% +- 58.70%
  Wilcoxon fixed vs rolling: p=0.0032

Graph stability across mid-training refits (mean edge Jaccard, 1.0=identical): 1.000 +- 0.000
Ties the rolling-window graph-stability analysis in Section 5.4 directly to whether
re-estimating mid-training changes realized trading performance.
OK  Saved: fixed_vs_rolling_scm.csv


## 16. Publication-Quality Figures (6 figures, PDF + PNG)

In [26]:
plt.rcParams.update({
    'figure.dpi': 300, 'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.fontsize': 9, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
})
PAL = {
    'causal_dqn'    : '#E63946', 'causal_sac'    : '#C1121F',
    'dueling_dqn'   : '#457B9D', 'discrete_sac'  : '#1D3557',
    'random_cf'     : '#F4A261', 'no_cql'         : '#2A9D8F',
    'no_per'        : '#8338EC', 'buy_hold'       : '#A8DADC',
    'momentum'      : '#6D6875', 'mean_reversion' : '#B5838D',
}
LAB = {
    'causal_dqn'    : 'Causal DQN (Ours)', 'causal_sac'   : 'Causal SAC (Ours)',
    'dueling_dqn'   : 'Dueling DQN',       'discrete_sac' : 'Discrete SAC',
    'random_cf'     : 'Random CF',          'no_cql'       : 'CER w/o CQL',
    'no_per'        : 'CER w/o PER',        'buy_hold'     : 'Buy-Hold',
    'momentum'      : 'Momentum',           'mean_reversion': 'Mean Reversion',
}

def savefig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight')
    plt.savefig(f'{name}.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f'OK  Saved: {name}.pdf/.png')

# Figure 1: Violin -- Return & Sharpe
plot_m = ['causal_dqn','causal_sac','dueling_dqn','discrete_sac','random_cf','buy_hold','momentum']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (metric, ylabel, src) in zip(axes, [
    ('total_return', 'Total Return', all_ret),
    ('sharpe_ratio', 'Sharpe Ratio', all_sharpe),
]):
    data_list = [src[m] for m in plot_m if src.get(m)]
    valid_m   = [m for m in plot_m if src.get(m)]
    parts = ax.violinplot(data_list, positions=range(len(valid_m)),
                          showmeans=True, showmedians=True)
    for pc, m in zip(parts['bodies'], valid_m):
        pc.set_facecolor(PAL[m]); pc.set_alpha(0.72)
    ax.set_xticks(range(len(valid_m)))
    ax.set_xticklabels([LAB[m] for m in valid_m], rotation=35, ha='right')
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.set_ylabel(ylabel); ax.grid(True, alpha=0.3, axis='y')

axes[0].set_title('Return Distribution', fontweight='bold')
axes[1].set_title('Sharpe Distribution', fontweight='bold')
plt.suptitle(f'CER-SCM vs Baselines -- {len(valid)} Assets, {N_TRIALS} Trials each',
             fontsize=12, y=1.01)
plt.tight_layout()
savefig('fig1_distributions')


OK  Saved: fig1_distributions.pdf/.png


In [27]:
# Figure 2: Ablation bars
abl_m   = list(ABLATION_LABELS.keys())
metrics = [('total_return', 'Total Return'), ('sharpe_ratio', 'Sharpe'),
           ('sortino_ratio', 'Sortino'),      ('calmar_ratio', 'Calmar')]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (met, title) in zip(axes, metrics):
    vals = [np.mean(flat_metric(m, met)) * (100 if met == 'total_return' else 1)
            for m in abl_m]
    colors = ['#E63946' if m == 'causal_dqn' else '#457B9D' for m in abl_m]
    bars = ax.bar(range(len(abl_m)), vals, color=colors, alpha=0.85)
    for bar in bars[:1]:
        bar.set_edgecolor('black'); bar.set_linewidth(2)
    ax.set_xticks(range(len(abl_m)))
    ax.set_xticklabels([ABLATION_LABELS[m] for m in abl_m], rotation=40, ha='right', fontsize=8)
    ax.axhline(0, color='gray', lw=0.6, ls='--')
    ax.set_title(title, fontweight='bold'); ax.grid(True, alpha=0.3, axis='y')
plt.suptitle('Ablation Study -- Component Contributions', fontsize=13, y=1.02)
plt.tight_layout()
savefig('fig2_ablation')

# Figure 3: Risk-Return scatter
fig, ax = plt.subplots(figsize=(10, 8))
for m in ALL_METHODS:
    if not all_ret.get(m) or not all_sharpe.get(m): continue
    ax.scatter(all_sharpe[m], [r * 100 for r in all_ret[m]],
               alpha=0.25, s=20, color=PAL[m], label=None)
    ax.scatter(np.mean(all_sharpe[m]), np.mean(all_ret[m]) * 100,
               s=180, marker='*', color=PAL[m], zorder=5, label=LAB[m])
ax.axhline(0, color='gray', lw=0.8, ls='--'); ax.axvline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Sharpe Ratio'); ax.set_ylabel('Total Return (%)')
ax.set_title('Risk-Return Frontier', fontweight='bold')
ax.legend(loc='upper left', framealpha=0.9); ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig('fig3_risk_return')

# Figure 4: Sector heatmap
heat_m    = ['causal_dqn','causal_sac','dueling_dqn','buy_hold']
sectors   = list(ASSET_UNIVERSE.keys())
heat_data = np.zeros((len(sectors), len(heat_m)))
for si, sec in enumerate(sectors):
    tickers_s = ASSET_UNIVERSE[sec]
    for mi, m in enumerate(heat_m):
        vals = []
        for t in tickers_s:
            if t not in valid: continue
            if m in METHODS_RL and results[t].get(m):
                vals.append(np.mean([r['total_return'] for r in results[t][m]]))
            elif m in METHODS_STATIC and isinstance(results[t].get(m), dict):
                vals.append(results[t][m]['total_return'])
        heat_data[si, mi] = np.mean(vals) * 100 if vals else 0.

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(heat_data, cmap='RdYlGn', aspect='auto',
               vmin=-max(abs(heat_data.min()), abs(heat_data.max())),
               vmax=max(abs(heat_data.min()), abs(heat_data.max())))
ax.set_xticks(range(len(heat_m))); ax.set_xticklabels([LAB[m] for m in heat_m], rotation=20)
ax.set_yticks(range(len(sectors))); ax.set_yticklabels(sectors)
for si in range(len(sectors)):
    for mi in range(len(heat_m)):
        ax.text(mi, si, f'{heat_data[si,mi]:+.1f}%', ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=ax, label='Mean Total Return (%)')
ax.set_title('Sector-Level Mean Total Return (%)', fontweight='bold')
plt.tight_layout()
savefig('fig4_sector_heatmap')

# Figure 5: Significance
comp_m = [m for m in ALL_METHODS if m != 'causal_dqn']
cd_vals = [c_ds.get(m, 0)    for m in comp_m]
lp_vals = [-np.log10(corrected.get(m, 1) + 1e-10) for m in comp_m]
colors  = ['#E63946' if corrected.get(m, 1) < 0.05 else '#AAAAAA' for m in comp_m]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(range(len(comp_m)), cd_vals,  color=colors, alpha=0.85)
axes[0].axvline(0, color='gray', lw=0.8); axes[0].axvline(0.5, color='orange', lw=1, ls='--')
axes[0].axvline(0.8, color='red', lw=1, ls='--')
axes[0].set_yticks(range(len(comp_m))); axes[0].set_yticklabels([LAB.get(m,m) for m in comp_m])
axes[0].set_xlabel("Cohen's d"); axes[0].set_title('Effect Size', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')
axes[1].barh(range(len(comp_m)), lp_vals, color=colors, alpha=0.85)
axes[1].axvline(-np.log10(0.05), color='orange', lw=1.5, ls='--', label='p=0.05')
axes[1].axvline(-np.log10(0.01), color='red',    lw=1.5, ls='--', label='p=0.01')
axes[1].set_yticks(range(len(comp_m))); axes[1].set_yticklabels([LAB.get(m,m) for m in comp_m])
axes[1].set_xlabel('-log10(Bonferroni p)'); axes[1].set_title('Statistical Significance', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='x')
plt.suptitle('CausalDQN vs All Baselines -- Effect Size and Significance', fontsize=12)
plt.tight_layout()
savefig('fig5_significance')

# Figure 6: Causal graph (AAPL)
try:
    import networkx as nx
    demo_g, demo_vn, _ = learn_causal_structure(
        TradingDataLoader('AAPL', TRAIN_START, TRAIN_END).load(), method='pc')
    G   = nx.DiGraph()
    G.add_nodes_from(demo_vn)
    for i in range(len(demo_vn)):
        for j in range(len(demo_vn)):
            if demo_g[i, j] != 0:
                G.add_edge(demo_vn[i], demo_vn[j])
    fig, ax = plt.subplots(figsize=(12, 8))
    pos = nx.spring_layout(G, seed=42, k=2.5)
    node_colors = ['#E63946' if n == 'action' else
                   '#457B9D' if n == 'returns' else '#A8DADC' for n in G.nodes()]
    nx.draw_networkx(G, pos, ax=ax, node_color=node_colors, node_size=1800,
                     font_size=8, font_weight='bold', arrows=True,
                     arrowsize=20, edge_color='#333333', width=1.5,
                     connectionstyle='arc3,rad=0.1')
    ax.set_title('Learned Causal Graph -- AAPL (PC algorithm, alpha=0.05)\n'
                 'Red=action, Blue=returns, Light blue=features', fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    savefig('fig6_causal_graph')
except Exception as e:
    print(f'Fig 6 skipped: {e}')


OK  Saved: fig2_ablation.pdf/.png


KeyError: 'causal_dqn_neural'

## 17. Final Summary -- Paper Claims

In [21]:
cr_ret = np.mean(all_ret['causal_dqn'])   if all_ret['causal_dqn'] else float('nan')
dd_ret = np.mean(all_ret['dueling_dqn'])  if all_ret['dueling_dqn'] else float('nan')
bh_ret = np.mean(all_ret['buy_hold'])     if all_ret['buy_hold'] else float('nan')
cr_sh  = np.mean(all_sharpe['causal_dqn']) if all_sharpe['causal_dqn'] else float('nan')
dd_sh  = np.mean(all_sharpe['dueling_dqn']) if all_sharpe['dueling_dqn'] else float('nan')

best_comp = max([m for m in ALL_METHODS if m != 'causal_dqn' and all_ret.get(m)],
                key=lambda m: np.mean(all_ret[m]))

print('='*70)
print('EXPERT SYSTEMS WITH APPLICATIONS -- SUBMISSION CLAIMS')
print('='*70)
print(f"""
Dataset   : {len(valid)} assets | 5 sectors | global markets (US, Asia, EU, Commodities)
Period    : Train {TRAIN_START}--{TRAIN_END} | Val {VAL_START}--{VAL_END} | Test {TEST_START}--{TEST_END}
Trials    : {N_TRIALS} per method (seeds 100--{100+N_TRIALS-1})  |  Episodes: {N_EPISODES}
Device    : {DEVICE}

Performance:
  Causal DQN (ours) : {cr_ret*100:+.2f}%  |  Sharpe {cr_sh:.3f}
  Dueling DQN       : {dd_ret*100:+.2f}%  |  Sharpe {dd_sh:.3f}
  Buy-Hold          : {bh_ret*100:+.2f}%
  Best competitor   : {best_comp} ({np.mean(all_ret[best_comp])*100:+.2f}%)

Improvements:
  vs Dueling DQN : {(cr_ret-dd_ret)/(abs(dd_ret)+1e-9)*100:+.1f}%
  vs Buy-Hold    : {(cr_ret-bh_ret)/(abs(bh_ret)+1e-9)*100:+.1f}%

Statistics (Wilcoxon + Bonferroni):
  vs Dueling DQN : p={corrected.get('dueling_dqn', float('nan')):.6f}  d={c_ds.get('dueling_dqn',0):.3f}
  vs Buy-Hold    : p={corrected.get('buy_hold', float('nan')):.6f}  d={c_ds.get('buy_hold',0):.3f}
  Friedman (RL)  : chi2={f_stat:.3f}  p={f_pval:.6f}

Saved outputs:
  journal_ablation.csv
  journal_full_results.csv
  fig1_distributions.pdf/png  -- violin plots
  fig2_ablation.pdf/png       -- ablation bar charts
  fig3_risk_return.pdf/png    -- risk-return frontier
  fig4_sector_heatmap.pdf/png -- sector heatmap
  fig5_significance.pdf/png   -- Cohen's d + Bonferroni p
  fig6_causal_graph.pdf/png   -- AAPL causal graph
""")


EXPERT SYSTEMS WITH APPLICATIONS -- SUBMISSION CLAIMS

Dataset   : 30 assets | 5 sectors | global markets (US, Asia, EU, Commodities)
Period    : Train 2018-01-01--2021-12-31 | Val 2022-01-01--2022-12-31 | Test 2023-01-01--2024-12-31
Trials    : 10 per method (seeds 100--109)  |  Episodes: 100
Device    : cuda

Performance:
  Causal DQN (ours) : +25.59%  |  Sharpe 0.481
  Dueling DQN       : +28.68%  |  Sharpe 0.552
  Buy-Hold          : +78.91%
  Best competitor   : buy_hold (+78.91%)

Improvements:
  vs Dueling DQN : -10.8%
  vs Buy-Hold    : -67.6%

Statistics (Wilcoxon + Bonferroni):
  vs Dueling DQN : p=0.830537  d=-0.056
  vs Buy-Hold    : p=0.000242  d=-0.746
  Friedman (RL)  : chi2=17.157  p=0.008723

Saved outputs:
  journal_ablation.csv
  journal_full_results.csv
  fig1_distributions.pdf/png  -- violin plots
  fig2_ablation.pdf/png       -- ablation bar charts
  fig3_risk_return.pdf/png    -- risk-return frontier
  fig4_sector_heatmap.pdf/png -- sector heatmap
  fig5_signific